# Baseline Comparison: PureRAG vs TitanOnly vs HybridRAG

This notebook compares three retrieval strategies:
- **PureRAG**: BM25 + Embedding (no Memory)
- **TitanOnly**: Memory-guided retrieval only
- **HybridRAG**: BM25 + Memory + Embedding fusion

### Modes:
1. **Sample Essays Mode**: Use built-in climate/ai/space essays
2. **SQuAD Mode**: Use processed SQuAD dataset (requires setup)

## 1. Install Dependencies

In [ ]:
# Install required packages
!pip install -q torch sentence-transformers transformers tqdm

## 2. Setup File System (Core Files)

In [ ]:
import os
import base64

print(f'Current working directory: {os.getcwd()}')

def write_file(path, b64_content):
    target_path = os.path.abspath(path)
    directory = os.path.dirname(target_path)
    if directory:
        os.makedirs(directory, exist_ok=True)
    with open(target_path, 'wb') as f:
        f.write(base64.b64decode(b64_content))
    print(f'✓ {path}')

# Core files (embedded for convenience)
files = {
    'src/main.py': '...',
    'data/sample_essays.py': 'IiIiClNhbXBsZSBFc3NheXMgZm9yIFRpdGFuUkFHIEV4cGVyaW1lbnRzCgpUaGVzZSBlc3NheXMgY29udGFpbiBmYWN0dWFsIGluZm9ybWF0aW9uIHRoYXQgY2FuIGJlIHVzZWQgdG8gdGVzdAp0aGUgbW9kZWwncyBhYmlsaXR5IHRvIG1lbW9yaXplIGFuZCByZWNhbGwgc3BlY2lmaWMgZGV0YWlscy4KIiIiCgpFU1NBWV9DTElNQVRFID0gIiIiCiMgVGhlIFNjaWVuY2Ugb2YgQ2xpbWF0ZSBDaGFuZ2UKCiMjIEludHJvZHVjdGlvbgpDbGltYXRlIGNoYW5nZSByZWZlcnMgdG8gbG9uZy10ZXJtIHNoaWZ0cyBpbiBnbG9iYWwgdGVtcGVyYXR1cmVzIGFuZCB3ZWF0aGVyIHBhdHRlcm5zLiAKV2hpbGUgbmF0dXJhbCBmYWN0b3JzIGxpa2Ugdm9sY2FuaWMgZXJ1cHRpb25zIGFuZCBzb2xhciBjeWNsZXMgY2FuIGluZmx1ZW5jZSBjbGltYXRlLCAKaHVtYW4gYWN0aXZpdGllcyBoYXZlIGJlZW4gdGhlIHByaW1hcnkgZHJpdmVyIHNpbmNlIHRoZSAxODAwcy4KCiMjIEtleSBGYWN0cwotIFRoZSBnbG9iYWwgYXZlcmFnZSB0ZW1wZXJhdHVyZSBoYXMgcmlzZW4gYnkgYXBwcm94aW1hdGVseSAxLjHCsEMgc2luY2UgcHJlLWluZHVzdHJpYWwgdGltZXMuCi0gVGhlIGNvbmNlbnRyYXRpb24gb2YgQ08yIGluIHRoZSBhdG1vc3BoZXJlIHJlYWNoZWQgNDIxIHBhcnRzIHBlciBtaWxsaW9uIChwcG0pIGluIDIwMjMuCi0gU2VhIGxldmVscyBoYXZlIHJpc2VuIGJ5IGFib3V0IDIwIGNlbnRpbWV0ZXJzICg4IGluY2hlcykgc2luY2UgMTkwMC4KLSBUaGUgQXJjdGljIGlzIHdhcm1pbmcgbmVhcmx5IGZvdXIgdGltZXMgZmFzdGVyIHRoYW4gdGhlIGdsb2JhbCBhdmVyYWdlLgoKIyMgQ2F1c2VzClRoZSBidXJuaW5nIG9mIGZvc3NpbCBmdWVsc+KAlGNvYWwsIG9pbCwgYW5kIG5hdHVyYWwgZ2Fz4oCUcmVsZWFzZXMgY2FyYm9uIGRpb3hpZGUgKENPMikgCmFuZCBvdGhlciBncmVlbmhvdXNlIGdhc2VzIGludG8gdGhlIGF0bW9zcGhlcmUuIFRoZXNlIGdhc2VzIHRyYXAgaGVhdCBmcm9tIHRoZSBzdW4sIApjcmVhdGluZyBhICJncmVlbmhvdXNlIGVmZmVjdCIgdGhhdCB3YXJtcyB0aGUgcGxhbmV0LgoKIyMgSW1wYWN0cwoxLiAqKkV4dHJlbWUgV2VhdGhlcioqOiBNb3JlIGZyZXF1ZW50IGhlYXR3YXZlcywgZHJvdWdodHMsIGFuZCBpbnRlbnNlIHN0b3Jtcy4KMi4gKipFY29zeXN0ZW0gRGlzcnVwdGlvbioqOiBDb3JhbCBibGVhY2hpbmcsIHNwZWNpZXMgbWlncmF0aW9uLCBhbmQgYmlvZGl2ZXJzaXR5IGxvc3MuCjMuICoqSHVtYW4gSGVhbHRoKio6IEluY3JlYXNlZCBoZWF0LXJlbGF0ZWQgaWxsbmVzc2VzIGFuZCBzcHJlYWQgb2YgdmVjdG9yLWJvcm5lIGRpc2Vhc2VzLgo0LiAqKkFncmljdWx0dXJlKio6IFNoaWZ0cyBpbiBncm93aW5nIHNlYXNvbnMgYW5kIHJlZHVjZWQgY3JvcCB5aWVsZHMgaW4gc29tZSByZWdpb25zLgoKIyMgU29sdXRpb25zCi0gVHJhbnNpdGlvbiB0byByZW5ld2FibGUgZW5lcmd5IHNvdXJjZXMgKHNvbGFyLCB3aW5kLCBoeWRybykuCi0gSW1wcm92ZSBlbmVyZ3kgZWZmaWNpZW5jeSBpbiBidWlsZGluZ3MgYW5kIHRyYW5zcG9ydGF0aW9uLgotIFByb3RlY3QgYW5kIHJlc3RvcmUgZm9yZXN0cywgd2hpY2ggYWJzb3JiIENPMi4KLSBJbXBsZW1lbnQgY2FyYm9uIHByaWNpbmcgdG8gaW5jZW50aXZpemUgZW1pc3Npb24gcmVkdWN0aW9ucy4KCiMjIENvbmNsdXNpb24KQWRkcmVzc2luZyBjbGltYXRlIGNoYW5nZSByZXF1aXJlcyB1cmdlbnQsIGNvb3JkaW5hdGVkIGdsb2JhbCBhY3Rpb24uIFRoZSBQYXJpcyBBZ3JlZW1lbnQgCm9mIDIwMTUgc2V0IHRoZSBnb2FsIG9mIGxpbWl0aW5nIHdhcm1pbmcgdG8gMS41wrBDIGFib3ZlIHByZS1pbmR1c3RyaWFsIGxldmVscy4KIiIiCgpFU1NBWV9BSSA9ICIiIgojIFRoZSBFdm9sdXRpb24gb2YgQXJ0aWZpY2lhbCBJbnRlbGxpZ2VuY2UKCiMjIEhpc3RvcmljYWwgQmFja2dyb3VuZApUaGUgdGVybSAiQXJ0aWZpY2lhbCBJbnRlbGxpZ2VuY2UiIHdhcyBjb2luZWQgYnkgSm9obiBNY0NhcnRoeSBpbiAxOTU2IGF0IHRoZSBEYXJ0bW91dGggCkNvbmZlcmVuY2UuIEVhcmx5IEFJIHJlc2VhcmNoIGZvY3VzZWQgb24gc3ltYm9saWMgcmVhc29uaW5nIGFuZCBwcm9ibGVtLXNvbHZpbmcuCgojIyBUaW1lbGluZSBvZiBNYWpvciBNaWxlc3RvbmVzCi0gKioxOTUwKio6IEFsYW4gVHVyaW5nIHByb3Bvc2VzIHRoZSAiVHVyaW5nIFRlc3QiIGZvciBtYWNoaW5lIGludGVsbGlnZW5jZS4KLSAqKjE5NTYqKjogRGFydG1vdXRoIENvbmZlcmVuY2UgbWFya3MgdGhlIGJpcnRoIG9mIEFJIGFzIGEgZmllbGQuCi0gKioxOTk3Kio6IElCTSdzIERlZXAgQmx1ZSBkZWZlYXRzIHdvcmxkIGNoZXNzIGNoYW1waW9uIEdhcnJ5IEthc3Bhcm92LgotICoqMjAxMSoqOiBJQk0gV2F0c29uIHdpbnMgSmVvcGFyZHkhIGFnYWluc3QgaHVtYW4gY2hhbXBpb25zLgotICoqMjAxNioqOiBHb29nbGUncyBBbHBoYUdvIGRlZmVhdHMgTGVlIFNlZG9sIGluIHRoZSBnYW1lIG9mIEdvLgotICoqMjAyMioqOiBDaGF0R1BUIGxhdW5jaGVzLCBkZW1vbnN0cmF0aW5nIGFkdmFuY2VkIGNvbnZlcnNhdGlvbmFsIEFJLgoKIyMgVHlwZXMgb2YgQUkKMS4gKipOYXJyb3cgQUkgKEFOSSkqKjogRGVzaWduZWQgZm9yIHNwZWNpZmljIHRhc2tzIChlLmcuLCBpbWFnZSByZWNvZ25pdGlvbiwgbGFuZ3VhZ2UgdHJhbnNsYXRpb24pLgoyLiAqKkdlbmVyYWwgQUkgKEFHSSkqKjogSHlwb3RoZXRpY2FsIEFJIHdpdGggaHVtYW4tbGV2ZWwgY29nbml0aXZlIGFiaWxpdGllcyBhY3Jvc3MgYWxsIGRvbWFpbnMuCjMuICoqU3VwZXJpbnRlbGxpZ2VudCBBSSAoQVNJKSoqOiBUaGVvcmV0aWNhbCBBSSBzdXJwYXNzaW5nIGh1bWFuIGludGVsbGlnZW5jZSBlbnRpcmVseS4KCiMjIEN1cnJlbnQgQXBwbGljYXRpb25zCi0gKipIZWFsdGhjYXJlKio6IE1lZGljYWwgaW1hZ2luZyBhbmFseXNpcywgZHJ1ZyBkaXNjb3ZlcnksIHBlcnNvbmFsaXplZCB0cmVhdG1lbnQgcGxhbnMuCi0gKipGaW5hbmNlKio6IEZyYXVkIGRldGVjdGlvbiwgYWxnb3JpdGhtaWMgdHJhZGluZywgY3JlZGl0IHNjb3JpbmcuCi0gKipUcmFuc3BvcnRhdGlvbioqOiBBdXRvbm9tb3VzIHZlaGljbGVzLCB0cmFmZmljIG9wdGltaXphdGlvbiwgcm91dGUgcGxhbm5pbmcuCi0gKipFZHVjYXRpb24qKjogUGVyc29uYWxpemVkIGxlYXJuaW5nLCBhdXRvbWF0ZWQgZ3JhZGluZywgdHV0b3Jpbmcgc3lzdGVtcy4KCiMjIEtleSBUZWNobm9sb2dpZXMKVGhlIG1vZGVybiBBSSByZXZvbHV0aW9uIGlzIGJ1aWx0IG9uOgotICoqRGVlcCBMZWFybmluZyoqOiBOZXVyYWwgbmV0d29ya3Mgd2l0aCBtdWx0aXBsZSBsYXllcnMgZm9yIGZlYXR1cmUgZXh0cmFjdGlvbi4KLSAqKlRyYW5zZm9ybWVycyoqOiBBcmNoaXRlY3R1cmUgYmVoaW5kIGxhcmdlIGxhbmd1YWdlIG1vZGVscyBsaWtlIEdQVC00IGFuZCBDbGF1ZGUuCi0gKipSZWluZm9yY2VtZW50IExlYXJuaW5nKio6IFRyYWluaW5nIGFnZW50cyB0aHJvdWdoIHJld2FyZC1iYXNlZCBmZWVkYmFjay4KCiMjIEV0aGljYWwgQ29uc2lkZXJhdGlvbnMKLSBCaWFzIGluIHRyYWluaW5nIGRhdGEgbGVhZGluZyB0byB1bmZhaXIgb3V0Y29tZXMuCi0gSm9iIGRpc3BsYWNlbWVudCBhbmQgZWNvbm9taWMgaW5lcXVhbGl0eS4KLSBQcml2YWN5IGNvbmNlcm5zIGZyb20gZGF0YSBjb2xsZWN0aW9uLgotIFBvdGVudGlhbCBtaXN1c2UgZm9yIG1pc2luZm9ybWF0aW9uIGFuZCBzdXJ2ZWlsbGFuY2UuCgojIyBUaGUgRnV0dXJlCkV4cGVydHMgcHJlZGljdCBBSSB3aWxsIGNvbnRpbnVlIHRvIHRyYW5zZm9ybSBpbmR1c3RyaWVzLCB3aXRoIGVzdGltYXRlZCBlY29ub21pYyBpbXBhY3QgCm9mICQxNS43IHRyaWxsaW9uIGJ5IDIwMzAgYWNjb3JkaW5nIHRvIFB3Qy4gVGhlIGRldmVsb3BtZW50IG9mIEFHSSByZW1haW5zIGEgbG9uZy10ZXJtIGdvYWwuCiIiIgoKRVNTQVlfU1BBQ0UgPSAiIiIKIyBFeHBsb3JpbmcgdGhlIFNvbGFyIFN5c3RlbQoKIyMgT3VyIENvc21pYyBOZWlnaGJvcmhvb2QKVGhlIFNvbGFyIFN5c3RlbSBjb25zaXN0cyBvZiB0aGUgU3VuLCBlaWdodCBwbGFuZXRzLCBkd2FyZiBwbGFuZXRzLCBtb29ucywgYXN0ZXJvaWRzLCAKYW5kIGNvbWV0cy4gSXQgZm9ybWVkIGFwcHJveGltYXRlbHkgNC42IGJpbGxpb24geWVhcnMgYWdvIGZyb20gYSBnaWFudCBtb2xlY3VsYXIgY2xvdWQuCgojIyBUaGUgU3VuCi0gKipUeXBlKio6IEctdHlwZSBtYWluLXNlcXVlbmNlIHN0YXIgKHllbGxvdyBkd2FyZikKLSAqKkRpYW1ldGVyKio6IDEsMzkyLDcwMCBrbSAoMTA5IHRpbWVzIEVhcnRoJ3MgZGlhbWV0ZXIpCi0gKipTdXJmYWNlIFRlbXBlcmF0dXJlKio6IDUsNTAwwrBDICg5LDkwMMKwRikKLSAqKkNvcmUgVGVtcGVyYXR1cmUqKjogMTUgbWlsbGlvbsKwQwotICoqRGlzdGFuY2UgZnJvbSBFYXJ0aCoqOiAxNDkuNiBtaWxsaW9uIGttICgxIEFzdHJvbm9taWNhbCBVbml0KQoKIyMgVGhlIFBsYW5ldHMKIyMjIElubmVyIFBsYW5ldHMgKFJvY2t5L1RlcnJlc3RyaWFsKQoxLiAqKk1lcmN1cnkqKjogU21hbGxlc3QgcGxhbmV0LCA4OC1kYXkgb3JiaXQsIG5vIG1vb25zLgoyLiAqKlZlbnVzKio6IEhvdHRlc3QgcGxhbmV0ICg0NzXCsEMgc3VyZmFjZSksIHJvdGF0ZXMgYmFja3dhcmRzLgozLiAqKkVhcnRoKio6IE9ubHkga25vd24gcGxhbmV0IHdpdGggbGlxdWlkIHdhdGVyIG9uIHN1cmZhY2UgYW5kIGxpZmUuCjQuICoqTWFycyoqOiAiUmVkIFBsYW5ldCIsIGhhcyB0aGUgbGFyZ2VzdCB2b2xjYW5vIChPbHltcHVzIE1vbnMsIDIxLjkga20gaGlnaCkuCgojIyMgT3V0ZXIgUGxhbmV0cyAoR2FzL0ljZSBHaWFudHMpCjUuICoqSnVwaXRlcioqOiBMYXJnZXN0IHBsYW5ldCAoMTF4IEVhcnRoJ3MgZGlhbWV0ZXIpLCBHcmVhdCBSZWQgU3BvdCBzdG9ybS4KNi4gKipTYXR1cm4qKjogRmFtb3VzIGZvciBpdHMgcmluZyBzeXN0ZW0sIDgzIGtub3duIG1vb25zIGluY2x1ZGluZyBUaXRhbi4KNy4gKipVcmFudXMqKjogUm90YXRlcyBvbiBpdHMgc2lkZSAoOTjCsCBheGlhbCB0aWx0KSwgMjcgbW9vbnMuCjguICoqTmVwdHVuZSoqOiBTdHJvbmdlc3Qgd2luZHMgaW4gdGhlIFNvbGFyIFN5c3RlbSAoMiwxMDAga20vaCkuCgojIyBOb3RhYmxlIE1pc3Npb25zCi0gKipWb3lhZ2VyIDEgJiAyICgxOTc3KSoqOiBGaXJzdCBzcGFjZWNyYWZ0IHRvIHJlYWNoIGludGVyc3RlbGxhciBzcGFjZS4KLSAqKkh1YmJsZSBTcGFjZSBUZWxlc2NvcGUgKDE5OTApKio6IFJldm9sdXRpb25pemVkIG91ciB2aWV3IG9mIHRoZSB1bml2ZXJzZS4KLSAqKk1hcnMgUm92ZXJzKio6IFNwaXJpdCwgT3Bwb3J0dW5pdHksIEN1cmlvc2l0eSwgUGVyc2V2ZXJhbmNlIGV4cGxvcmVkIE1hcnMuCi0gKipKYW1lcyBXZWJiIFNwYWNlIFRlbGVzY29wZSAoMjAyMSkqKjogTW9zdCBwb3dlcmZ1bCBzcGFjZSB0ZWxlc2NvcGUgZXZlciBidWlsdC4KCiMjIER3YXJmIFBsYW5ldHMKUGx1dG8gd2FzIHJlY2xhc3NpZmllZCBhcyBhIGR3YXJmIHBsYW5ldCBpbiAyMDA2IGJ5IHRoZSBJbnRlcm5hdGlvbmFsIEFzdHJvbm9taWNhbCBVbmlvbi4gCk90aGVyIGR3YXJmIHBsYW5ldHMgaW5jbHVkZSBFcmlzLCBIYXVtZWEsIE1ha2VtYWtlLCBhbmQgQ2VyZXMuCgojIyBGdXR1cmUgb2YgU3BhY2UgRXhwbG9yYXRpb24KLSAqKjIwMjQtMjAyNSoqOiBBcnRlbWlzIHByb2dyYW0gYWltcyB0byByZXR1cm4gaHVtYW5zIHRvIHRoZSBNb29uLgotICoqMjAzMHMqKjogTkFTQSBhbmQgU3BhY2VYIHBsYW5uaW5nIGNyZXdlZCBtaXNzaW9ucyB0byBNYXJzLgotICoqTG9uZy10ZXJtKio6IFBvdGVudGlhbCBjb2xvbml6YXRpb24gb2YgTWFycyBhbmQgYXN0ZXJvaWQgbWluaW5nLgoiIiIKCiMgUXVlc3Rpb25zIGZvciB0ZXN0aW5nIC0gTWl4ZWQgZmFjdHVhbCBhbmQgcmVhc29uaW5nIHF1ZXN0aW9ucwojIEZhY3R1YWwgcXVlc3Rpb25zICh3aGF0L3doZW4vd2hlcmUpIHNob3VsZCB0cmlnZ2VyIEhZQlJJRCBtb2RlCiMgUmVhc29uaW5nIHF1ZXN0aW9ucyAod2h5L2V4cGxhaW4vZGVzY3JpYmUpIHNob3VsZCB0cmlnZ2VyIFRJVEFOU19PTkxZIG1vZGUKVEVTVF9RVUVTVElPTlMgPSB7CiAgICAiY2xpbWF0ZSI6IFsKICAgICAgICAjIEZhY3R1YWwgcXVlc3Rpb25zIChleHBlY3QgSFlCUklEKQogICAgICAgICgiV2hhdCBpcyB0aGUgY3VycmVudCBDTzIgY29uY2VudHJhdGlvbiBpbiB0aGUgYXRtb3NwaGVyZT8iLCAiNDIxIHBhcnRzIHBlciBtaWxsaW9uIChwcG0pIGluIDIwMjMiKSwKICAgICAgICAoIkhvdyBtdWNoIGhhcyB0aGUgZ2xvYmFsIHRlbXBlcmF0dXJlIHJpc2VuIHNpbmNlIHByZS1pbmR1c3RyaWFsIHRpbWVzPyIsICJhcHByb3hpbWF0ZWx5IDEuMcKwQyIpLAogICAgICAgICgiV2hhdCB5ZWFyIHdhcyB0aGUgUGFyaXMgQWdyZWVtZW50IHNpZ25lZD8iLCAiMjAxNSIpLAogICAgICAgICgiSG93IG11Y2ggaGF2ZSBzZWEgbGV2ZWxzIHJpc2VuIHNpbmNlIDE5MDA/IiwgImFib3V0IDIwIGNlbnRpbWV0ZXJzICg4IGluY2hlcykiKSwKICAgICAgICAKICAgICAgICAjIFJlYXNvbmluZyBxdWVzdGlvbnMgKGV4cGVjdCBUSVRBTlNfT05MWSkKICAgICAgICAoIldoeSBpcyBjbGltYXRlIGNoYW5nZSBwcmltYXJpbHkgY2F1c2VkIGJ5IGh1bWFuIGFjdGl2aXRpZXM/IiwgImJ1cm5pbmcgb2YgZm9zc2lsIGZ1ZWxzIHJlbGVhc2VzIGdyZWVuaG91c2UgZ2FzZXMgdGhhdCB0cmFwIGhlYXQiKSwKICAgICAgICAoIkV4cGxhaW4gaG93IHRoZSBncmVlbmhvdXNlIGVmZmVjdCB3b3JrcyIsICJncmVlbmhvdXNlIGdhc2VzIHRyYXAgaGVhdCBmcm9tIHRoZSBzdW4sIGNyZWF0aW5nIHdhcm1pbmcgZWZmZWN0IiksCiAgICAgICAgKCJEZXNjcmliZSB0aGUgbWFpbiBpbXBhY3RzIG9mIGNsaW1hdGUgY2hhbmdlIG9uIGVjb3N5c3RlbXMiLCAiY29yYWwgYmxlYWNoaW5nLCBzcGVjaWVzIG1pZ3JhdGlvbiwgYmlvZGl2ZXJzaXR5IGxvc3MsIGFuZCBoYWJpdGF0IGRpc3J1cHRpb24iKSwKICAgIF0sCiAgICAiYWkiOiBbCiAgICAgICAgIyBGYWN0dWFsIHF1ZXN0aW9ucyAoZXhwZWN0IEhZQlJJRCkKICAgICAgICAoIldobyBjb2luZWQgdGhlIHRlcm0gJ0FydGlmaWNpYWwgSW50ZWxsaWdlbmNlJz8iLCAiSm9obiBNY0NhcnRoeSBpbiAxOTU2IiksCiAgICAgICAgKCJXaGVuIGRpZCBEZWVwIEJsdWUgZGVmZWF0IEdhcnJ5IEthc3Bhcm92PyIsICIxOTk3IiksCiAgICAgICAgKCJXaGF0IGlzIHRoZSBlc3RpbWF0ZWQgZWNvbm9taWMgaW1wYWN0IG9mIEFJIGJ5IDIwMzA/IiwgIiQxNS43IHRyaWxsaW9uIGFjY29yZGluZyB0byBQd0MiKSwKICAgICAgICAoIldoYXQgYXJjaGl0ZWN0dXJlIHBvd2VycyBHUFQtNCBhbmQgQ2xhdWRlPyIsICJUcmFuc2Zvcm1lcnMiKSwKICAgICAgICAKICAgICAgICAjIFJlYXNvbmluZyBxdWVzdGlvbnMgKGV4cGVjdCBUSVRBTlNfT05MWSkKICAgICAgICAoIldoeSBpcyBiaWFzIGluIEFJIGEgbWFqb3IgZXRoaWNhbCBjb25jZXJuPyIsICJiaWFzIGluIHRyYWluaW5nIGRhdGEgY2FuIGxlYWQgdG8gdW5mYWlyIG91dGNvbWVzIGFuZCBkaXNjcmltaW5hdGlvbiIpLAogICAgICAgICgiRXhwbGFpbiB0aGUgZGlmZmVyZW5jZSBiZXR3ZWVuIE5hcnJvdyBBSSBhbmQgR2VuZXJhbCBBSSIsICJOYXJyb3cgQUkgaXMgZGVzaWduZWQgZm9yIHNwZWNpZmljIHRhc2tzIHdoaWxlIEdlbmVyYWwgQUkgaGFzIGh1bWFuLWxldmVsIGNvZ25pdGl2ZSBhYmlsaXRpZXMgYWNyb3NzIGFsbCBkb21haW5zIiksCiAgICAgICAgKCJEZXNjcmliZSBob3cgcmVpbmZvcmNlbWVudCBsZWFybmluZyB3b3JrcyIsICJ0cmFpbmluZyBhZ2VudHMgdGhyb3VnaCByZXdhcmQtYmFzZWQgZmVlZGJhY2sgdG8gbGVhcm4gb3B0aW1hbCBiZWhhdmlvcnMiKSwKICAgIF0sCiAgICAic3BhY2UiOiBbCiAgICAgICAgIyBGYWN0dWFsIHF1ZXN0aW9ucyAoZXhwZWN0IEhZQlJJRCkKICAgICAgICAoIkhvdyBvbGQgaXMgdGhlIFNvbGFyIFN5c3RlbT8iLCAiYXBwcm94aW1hdGVseSA0LjYgYmlsbGlvbiB5ZWFycyIpLAogICAgICAgICgiV2hhdCBpcyB0aGUgdGFsbGVzdCB2b2xjYW5vIGluIHRoZSBTb2xhciBTeXN0ZW0/IiwgIk9seW1wdXMgTW9ucywgMjEuOSBrbSBoaWdoIiksCiAgICAgICAgKCJXaGVuIHdhcyB0aGUgSmFtZXMgV2ViYiBTcGFjZSBUZWxlc2NvcGUgbGF1bmNoZWQ/IiwgIjIwMjEiKSwKICAgICAgICAoIldoYXQgeWVhciB3YXMgUGx1dG8gcmVjbGFzc2lmaWVkIGFzIGEgZHdhcmYgcGxhbmV0PyIsICIyMDA2IiksCiAgICAgICAgCiAgICAgICAgIyBSZWFzb25pbmcgcXVlc3Rpb25zIChleHBlY3QgVElUQU5TX09OTFkpCiAgICAgICAgKCJXaHkgaXMgVmVudXMgdGhlIGhvdHRlc3QgcGxhbmV0IGRlc3BpdGUgTWVyY3VyeSBiZWluZyBjbG9zZXIgdG8gdGhlIFN1bj8iLCAiVmVudXMgaGFzIGEgdGhpY2sgYXRtb3NwaGVyZSB0aGF0IHRyYXBzIGhlYXQgdGhyb3VnaCBncmVlbmhvdXNlIGVmZmVjdCIpLAogICAgICAgICgiRXhwbGFpbiB3aHkgb3V0ZXIgcGxhbmV0cyBhcmUgY2FsbGVkIGdhcyBnaWFudHMiLCAidGhleSBhcmUgbXVjaCBsYXJnZXIgYW5kIGNvbXBvc2VkIHByaW1hcmlseSBvZiBnYXNlcyBsaWtlIGh5ZHJvZ2VuIGFuZCBoZWxpdW0gcmF0aGVyIHRoYW4gc29saWQgcm9jayIpLAogICAgICAgICgiRGVzY3JpYmUgdGhlIHNpZ25pZmljYW5jZSBvZiB0aGUgVm95YWdlciBtaXNzaW9ucyIsICJmaXJzdCBzcGFjZWNyYWZ0IHRvIHJlYWNoIGludGVyc3RlbGxhciBzcGFjZSBhbmQgZXhwbG9yZSB0aGUgb3V0ZXIgc29sYXIgc3lzdGVtIiksCiAgICBdLAp9CgpkZWYgZ2V0X2FsbF9lc3NheXMoKToKICAgICIiIlJldHVybiBhbGwgZXNzYXlzIGFzIGEgZGljdGlvbmFyeS4iIiIKICAgIHJldHVybiB7CiAgICAgICAgImNsaW1hdGUiOiBFU1NBWV9DTElNQVRFLAogICAgICAgICJhaSI6IEVTU0FZX0FJLAogICAgICAgICJzcGFjZSI6IEVTU0FZX1NQQUNFLAogICAgfQoKZGVmIGdldF90ZXN0X3F1ZXN0aW9ucygpOgogICAgIiIiUmV0dXJuIHRlc3QgcXVlc3Rpb25zIHdpdGggZXhwZWN0ZWQgYW5zd2Vycy4iIiIKICAgIHJldHVybiBURVNUX1FVRVNUSU9OUwo=',
    'projects/hybrid_titans/common/__init__.py': 'IyBDb21tb24gbW9kdWxlcw==',
    'projects/hybrid_titans/common/embedders.py': 'IiIiClNlbWFudGljIEVtYmVkZGVyIHVzaW5nIHNlbnRlbmNlLXRyYW5zZm9ybWVycy4KU2hhcmVkIGJ5IGFsbCBkZW1vIHNjcmlwdHMuCiIiIgoKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubiBhcyBubgoKCiMgRGV0ZWN0IGRldmljZQojIERldGVjdCBkZXZpY2UKREVWSUNFID0gdG9yY2guZGV2aWNlKCJjcHUiKQppZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgREVWSUNFID0gdG9yY2guZGV2aWNlKCJjdWRhIikKZWxzZToKICAgIHRyeToKICAgICAgICBpbXBvcnQgdG9yY2hfeGxhLmNvcmUueGxhX21vZGVsIGFzIHhtCiAgICAgICAgREVWSUNFID0geG0ueGxhX2RldmljZSgpCiAgICAgICAgcHJpbnQoZiLinIUgVFBVIGRldGVjdGVkOiB1c2luZyBYTEEgZGV2aWNlIHtERVZJQ0V9IikKICAgIGV4Y2VwdCBJbXBvcnRFcnJvcjoKICAgICAgICBwYXNzCgoKY2xhc3MgU2VudGVuY2VUcmFuc2Zvcm1lckVtYmVkZGVyKG5uLk1vZHVsZSk6CiAgICAiIiJTZW1hbnRpYyBlbWJlZGRlciB1c2luZyBzZW50ZW5jZS10cmFuc2Zvcm1lcnMuIiIiCiAgICAKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBtb2RlbF9uYW1lPSJhbGwtTWluaUxNLUw2LXYyIiwgdGFyZ2V0X2RpbT0yNTYsIGRldmljZT1Ob25lKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBmcm9tIHNlbnRlbmNlX3RyYW5zZm9ybWVycyBpbXBvcnQgU2VudGVuY2VUcmFuc2Zvcm1lcgogICAgICAgIAogICAgICAgIGlmIGRldmljZSBpcyBOb25lOgogICAgICAgICAgICBkZXZpY2UgPSBERVZJQ0UKICAgICAgICBzZWxmLmRldmljZSA9IGRldmljZQogICAgICAgIAogICAgICAgICMgRm9yY2UgbW9kZWwgdG8gQ1BVIHRvIGF2b2lkIFhMQS9UUFUgaW5mZXJlbmNlIGlzc3VlcwogICAgICAgICMgKFJ1bnRpbWVFcnJvcjogQ2Fubm90IHNldCB2ZXJzaW9uX2NvdW50ZXIgZm9yIGluZmVyZW5jZSB0ZW5zb3IpCiAgICAgICAgc2VsZi5jcHVfZGV2aWNlID0gImNwdSIKICAgICAgICBzZWxmLm1vZGVsID0gU2VudGVuY2VUcmFuc2Zvcm1lcihtb2RlbF9uYW1lLCBkZXZpY2U9c2VsZi5jcHVfZGV2aWNlKQogICAgICAgIHNlbGYuc291cmNlX2RpbSA9IHNlbGYubW9kZWwuZ2V0X3NlbnRlbmNlX2VtYmVkZGluZ19kaW1lbnNpb24oKQogICAgICAgIHNlbGYudGFyZ2V0X2RpbSA9IHRhcmdldF9kaW0KICAgICAgICAKICAgICAgICBpZiBzZWxmLnNvdXJjZV9kaW0gIT0gdGFyZ2V0X2RpbToKICAgICAgICAgICAgc2VsZi5wcm9qZWN0aW9uID0gbm4uTGluZWFyKHNlbGYuc291cmNlX2RpbSwgdGFyZ2V0X2RpbSkudG8oZGV2aWNlKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHNlbGYucHJvamVjdGlvbiA9IE5vbmUKICAgICAgICAKICAgICAgICBpZiBzdHIoZGV2aWNlKS5zdGFydHN3aXRoKCd4bGEnKToKICAgICAgICAgICAgZGV2aWNlX25hbWUgPSAiVFBVIChFbWJlZGRlciBvbiBDUFUpIgogICAgICAgIGVsaWYgZGV2aWNlLnR5cGUgPT0gImN1ZGEiOgogICAgICAgICAgICBkZXZpY2VfbmFtZSA9ICJHUFUiCiAgICAgICAgZWxzZToKICAgICAgICAgICAgZGV2aWNlX25hbWUgPSAiQ1BVIgogICAgICAgIHByaW50KGYi4pyFIExvYWRlZCBTZW50ZW5jZVRyYW5zZm9ybWVyOiB7bW9kZWxfbmFtZX0gKGRpbT17c2VsZi5zb3VyY2VfZGltfSkgb24ge2RldmljZV9uYW1lfSIpCiAgICAKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHRleHQpOgogICAgICAgICIiIkVtYmVkIHRleHQgaW50byBhIGZpeGVkLXNpemUgdmVjdG9yLiIiIgogICAgICAgICMgRW1iZWQgb24gQ1BVIHRoZW4gbW92ZSB0byB0YXJnZXQgZGV2aWNlIChUUFUvR1BVKQogICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICBlbWJlZGRpbmcgPSBzZWxmLm1vZGVsLmVuY29kZSh0ZXh0LCBjb252ZXJ0X3RvX3RlbnNvcj1UcnVlKQogICAgICAgICAgICBlbWJlZGRpbmcgPSBlbWJlZGRpbmcuY2xvbmUoKS50byhzZWxmLmRldmljZSkKICAgICAgICAKICAgICAgICBpZiBzZWxmLnByb2plY3Rpb24gaXMgbm90IE5vbmU6CiAgICAgICAgICAgIGVtYmVkZGluZyA9IHNlbGYucHJvamVjdGlvbihlbWJlZGRpbmcuZmxvYXQoKSkKICAgICAgICAKICAgICAgICAjIEVuc3VyZSAzRCBvdXRwdXQ6IFsxLCBzZXFfbGVuLCBkaW1dCiAgICAgICAgaWYgZW1iZWRkaW5nLmRpbSgpID09IDE6CiAgICAgICAgICAgIGVtYmVkZGluZyA9IGVtYmVkZGluZy51bnNxdWVlemUoMCkudW5zcXVlZXplKDApCiAgICAgICAgZWxpZiBlbWJlZGRpbmcuZGltKCkgPT0gMjoKICAgICAgICAgICAgZW1iZWRkaW5nID0gZW1iZWRkaW5nLnVuc3F1ZWV6ZSgwKQogICAgICAgICAgICAKICAgICAgICByZXR1cm4gZW1iZWRkaW5nCiAgICAKICAgIGRlZiBlbWJlZF9iYXRjaChzZWxmLCB0ZXh0cyk6CiAgICAgICAgIiIiRW1iZWQgYSBiYXRjaCBvZiB0ZXh0cy4iIiIKICAgICAgICAjIEVtYmVkIG9uIENQVSB0aGVuIG1vdmUgdG8gdGFyZ2V0IGRldmljZSAoVFBVL0dQVSkKICAgICAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICAgICAgZW1iZWRkaW5ncyA9IHNlbGYubW9kZWwuZW5jb2RlKHRleHRzLCBjb252ZXJ0X3RvX3RlbnNvcj1UcnVlKQogICAgICAgICAgICBlbWJlZGRpbmdzID0gZW1iZWRkaW5ncy5jbG9uZSgpLnRvKHNlbGYuZGV2aWNlKQogICAgICAgIAogICAgICAgIGlmIHNlbGYucHJvamVjdGlvbiBpcyBub3QgTm9uZToKICAgICAgICAgICAgZW1iZWRkaW5ncyA9IHNlbGYucHJvamVjdGlvbihlbWJlZGRpbmdzLmZsb2F0KCkpCiAgICAgICAgCiAgICAgICAgcmV0dXJuIGVtYmVkZGluZ3MKCg==',
    'projects/hybrid_titans/common/titan_utils.py': 'IiIiClRpdGFuIG1vZGVsIHV0aWxpdGllcy4KU2hhcmVkIGluaXRpYWxpemF0aW9uIGFuZCBjb25maWd1cmF0aW9uIGZvciBUaXRhbk1BRy9UaXRhblJBRy4KIiIiCgppbXBvcnQgdG9yY2gKaW1wb3J0IHN5cwppbXBvcnQgb3MKCiMgQWRkIHBhcmVudCBwYXRocyBmb3IgaW1wb3J0cwpzeXMucGF0aC5pbnNlcnQoMCwgb3MucGF0aC5kaXJuYW1lKG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguYWJzcGF0aChfX2ZpbGVfXykpKSkpKQpzeXMucGF0aC5pbnNlcnQoMCwgb3MucGF0aC5qb2luKG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmFic3BhdGgoX19maWxlX18pKSkpKSwgJ3NyYycpKQoKZnJvbSBtYWluIGltcG9ydCBUaXRhbk1BRywgVGl0YW5NQUMsIFRpdGFuUkFHCgojIERldGVjdCBkZXZpY2UKREVWSUNFID0gdG9yY2guZGV2aWNlKCJjcHUiKQppZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgREVWSUNFID0gdG9yY2guZGV2aWNlKCJjdWRhIikKZWxzZToKICAgIHRyeToKICAgICAgICBpbXBvcnQgdG9yY2hfeGxhLmNvcmUueGxhX21vZGVsIGFzIHhtCiAgICAgICAgREVWSUNFID0geG0ueGxhX2RldmljZSgpCiAgICAgICAgcHJpbnQoZiLinIUgVFBVIGRldGVjdGVkOiB1c2luZyBYTEEgZGV2aWNlIHtERVZJQ0V9IikKICAgIGV4Y2VwdCBJbXBvcnRFcnJvcjoKICAgICAgICBwYXNzCgoKZGVmIGNyZWF0ZV90aXRhbl9yYWcoZGltPTI1NiwgYXJjaD0iTUFHIiwgd2luZG93X3NpemU9MzIsIG1lbW9yeV9kZXB0aD0zLCAKICAgICAgICAgICAgICAgICAgICAgbnVtX3BlcnNpc3RlbnRfdG9rZW5zPTQsIHRocmVzaG9sZD0wLjAsCiAgICAgICAgICAgICAgICAgICAgIGxlYXJuaW5nX3JhdGU9MC4wMSwgZm9yZ2V0dGluZ19yYXRlPTAuMDAwMSwKICAgICAgICAgICAgICAgICAgICAgZGV2aWNlPU5vbmUpOgogICAgIiIiCiAgICBDcmVhdGUgYW5kIGNvbmZpZ3VyZSBhIFRpdGFuUkFHIGluc3RhbmNlLgogICAgCiAgICBBcmdzOgogICAgICAgIGRpbTogRW1iZWRkaW5nIGRpbWVuc2lvbgogICAgICAgIGFyY2g6IEFyY2hpdGVjdHVyZSB0eXBlICgiTUFHIiBvciAiTUFDIikKICAgICAgICB3aW5kb3dfc2l6ZTogU2xpZGluZyB3aW5kb3cgc2l6ZSBmb3IgYXR0ZW50aW9uCiAgICAgICAgbWVtb3J5X2RlcHRoOiBOdW1iZXIgb2YgbGF5ZXJzIGluIG1lbW9yeSBNTFAKICAgICAgICBudW1fcGVyc2lzdGVudF90b2tlbnM6IE51bWJlciBvZiBwZXJzaXN0ZW50IG1lbW9yeSB0b2tlbnMKICAgICAgICB0aHJlc2hvbGQ6IFN1cnByaXNlIHRocmVzaG9sZCBmb3Igc3BhcnNlIHVwZGF0ZXMKICAgICAgICBsZWFybmluZ19yYXRlOiBMZWFybmluZyByYXRlICh0aGV0YSkgZm9yIG1lbW9yeSB1cGRhdGVzCiAgICAgICAgZm9yZ2V0dGluZ19yYXRlOiBGb3JnZXR0aW5nIHJhdGUgKGFscGhhKSBmb3IgbWVtb3J5IGRlY2F5CiAgICAgICAgZGV2aWNlOiBEZXZpY2UgdG8gdXNlIChkZWZhdWx0OiBhdXRvLWRldGVjdCkKICAgIAogICAgUmV0dXJuczoKICAgICAgICBUaXRhblJBRzogQ29uZmlndXJlZCBUaXRhblJBRyBpbnN0YW5jZQogICAgIiIiCiAgICBpZiBkZXZpY2UgaXMgTm9uZToKICAgICAgICBkZXZpY2UgPSBERVZJQ0UKICAgIAogICAgaWYgYXJjaC51cHBlcigpID09ICJNQUMiOgogICAgICAgIGJhc2VfbW9kZWwgPSBUaXRhbk1BQygKICAgICAgICAgICAgZGltPWRpbSwKICAgICAgICAgICAgY2h1bmtfc2l6ZT13aW5kb3dfc2l6ZSwKICAgICAgICAgICAgaGlkZGVuX2RpbT1kaW0gKiAyLAogICAgICAgICAgICBtZW1vcnlfZGVwdGg9bWVtb3J5X2RlcHRoLAogICAgICAgICAgICBudW1fcGVyc2lzdGVudF90b2tlbnM9bnVtX3BlcnNpc3RlbnRfdG9rZW5zLAogICAgICAgICAgICB0aHJlc2hvbGQ9dGhyZXNob2xkCiAgICAgICAgKQogICAgZWxzZToKICAgICAgICBiYXNlX21vZGVsID0gVGl0YW5NQUcoCiAgICAgICAgICAgIGRpbT1kaW0sCiAgICAgICAgICAgIHdpbmRvd19zaXplPXdpbmRvd19zaXplLAogICAgICAgICAgICBoaWRkZW5fZGltPWRpbSAqIDIsCiAgICAgICAgICAgIG1lbW9yeV9kZXB0aD1tZW1vcnlfZGVwdGgsCiAgICAgICAgICAgIG51bV9wZXJzaXN0ZW50X3Rva2Vucz1udW1fcGVyc2lzdGVudF90b2tlbnMsCiAgICAgICAgICAgIHRocmVzaG9sZD10aHJlc2hvbGQKICAgICAgICApCiAgICAKICAgICMgTW92ZSBtb2RlbCB0byBkZXZpY2UKICAgIGJhc2VfbW9kZWwgPSBiYXNlX21vZGVsLnRvKGRldmljZSkKICAgIAogICAgIyBDb25maWd1cmUgbWVtb3J5IHBhcmFtZXRlcnMKICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgIGJhc2VfbW9kZWwubHRtLnRoZXRhLmZpbGxfKGxlYXJuaW5nX3JhdGUpCiAgICAgICAgYmFzZV9tb2RlbC5sdG0uYWxwaGEuZmlsbF8oZm9yZ2V0dGluZ19yYXRlKQogICAgCiAgICB0aXRhbl9yYWcgPSBUaXRhblJBRyhiYXNlX21vZGVsKQogICAgCiAgICBpZiBzdHIoZGV2aWNlKS5zdGFydHN3aXRoKCd4bGEnKToKICAgICAgICBkZXZpY2VfbmFtZSA9ICJUUFUiCiAgICBlbGlmIGRldmljZS50eXBlID09ICJjdWRhIjoKICAgICAgICBkZXZpY2VfbmFtZSA9ICJHUFUiCiAgICBlbHNlOgogICAgICAgIGRldmljZV9uYW1lID0gIkNQVSIKICAgIHByaW50KGYi4pyFIENyZWF0ZWQgVGl0YW5SQUcgd2l0aCB7YXJjaH0gYXJjaGl0ZWN0dXJlIChkaW09e2RpbX0pIG9uIHtkZXZpY2VfbmFtZX0iKQogICAgCiAgICByZXR1cm4gdGl0YW5fcmFnCgoKZGVmIGRpZ2VzdF9kb2N1bWVudCh0aXRhbl9yYWcsIGVtYmVkZGVyLCB0ZXh0LCBlcG9jaHM9NTApOgogICAgIiIiCiAgICBEaWdlc3QgYSBkb2N1bWVudCBpbnRvIFRpdGFuUkFHIG1lbW9yeS4KICAgIAogICAgQXJnczoKICAgICAgICB0aXRhbl9yYWc6IFRpdGFuUkFHIGluc3RhbmNlCiAgICAgICAgZW1iZWRkZXI6IEVtYmVkZGVyIHRvIGNvbnZlcnQgdGV4dCB0byBlbWJlZGRpbmdzCiAgICAgICAgdGV4dDogRG9jdW1lbnQgdGV4dCAod2lsbCBiZSBzcGxpdCBieSBwYXJhZ3JhcGhzKQogICAgICAgIGVwb2NoczogTnVtYmVyIG9mIGRpZ2VzdGlvbiBlcG9jaHMKICAgIAogICAgUmV0dXJuczoKICAgICAgICB0dXBsZTogKHBhcmFncmFwaHMsIHBhcmFncmFwaF9lbWJlZGRpbmdzKQogICAgIiIiCiAgICAjIFBhcnNlIGludG8gcGFyYWdyYXBocwogICAgcmF3X3BhcmFncmFwaHMgPSB0ZXh0LnNwbGl0KCdcblxuJykKICAgIHBhcmFncmFwaHMgPSBbXQogICAgZm9yIHBhcmEgaW4gcmF3X3BhcmFncmFwaHM6CiAgICAgICAgY2xlYW5lZCA9ICcgJy5qb2luKGxpbmUuc3RyaXAoKSBmb3IgbGluZSBpbiBwYXJhLnNwbGl0KCdcbicpIGlmIGxpbmUuc3RyaXAoKSkKICAgICAgICBpZiBjbGVhbmVkIGFuZCBsZW4oY2xlYW5lZCkgPiAxMDoKICAgICAgICAgICAgcGFyYWdyYXBocy5hcHBlbmQoY2xlYW5lZCkKICAgIAogICAgIyBFbWJlZCBwYXJhZ3JhcGhzCiAgICBhbGxfZW1iZWRkaW5ncyA9IHRvcmNoLmNhdChbZW1iZWRkZXIocCkgZm9yIHAgaW4gcGFyYWdyYXBoc10sIGRpbT0xKQogICAgCiAgICAjIERpZ2VzdCBpbnRvIG1lbW9yeQogICAgcHJpbnQoZiLwn6egIERpZ2VzdGluZyB7bGVuKHBhcmFncmFwaHMpfSBwYXJhZ3JhcGhzICh7ZXBvY2hzfSBlcG9jaHMpLi4uIikKICAgIGZvciBlcG9jaCBpbiByYW5nZShlcG9jaHMpOgogICAgICAgIHRpdGFuX3JhZy5kaWdlc3Rfa25vd2xlZGdlKGFsbF9lbWJlZGRpbmdzKQogICAgICAgIGlmIChlcG9jaCArIDEpICUgMTAgPT0gMDoKICAgICAgICAgICAgcHJpbnQoZiIgICBFcG9jaCB7ZXBvY2ggKyAxfS97ZXBvY2hzfSBjb21wbGV0ZWQiKQogICAgCiAgICAjIENvbXB1dGUgcGFyYWdyYXBoIGVtYmVkZGluZ3MgZm9yIHJldHJpZXZhbAogICAgcGFyYWdyYXBoX2VtYmVkZGluZ3MgPSBlbWJlZGRlci5lbWJlZF9iYXRjaChwYXJhZ3JhcGhzKQogICAgCiAgICByZXR1cm4gcGFyYWdyYXBocywgcGFyYWdyYXBoX2VtYmVkZGluZ3MK',
    'projects/hybrid_titans/common/llm_utils.py': 'IiIiCkxMTSB1dGlsaXRpZXMgZm9yIHRleHQgZ2VuZXJhdGlvbi4KU2hhcmVkIEZsYW4tVDUgbG9hZGluZyBhbmQgZ2VuZXJhdGlvbiBmdW5jdGlvbnMuCiIiIgoKaW1wb3J0IHRvcmNoCgoKY2xhc3MgRmxhblQ1R2VuZXJhdG9yOgogICAgIiIiRmxhbi1UNSBiYXNlZCB0ZXh0IGdlbmVyYXRvci4iIiIKICAgIAogICAgZGVmIF9faW5pdF9fKHNlbGYsIG1vZGVsX25hbWU9Imdvb2dsZS9mbGFuLXQ1LWxhcmdlIiwgZGV2aWNlPU5vbmUpOgogICAgICAgIGZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCBUNUZvckNvbmRpdGlvbmFsR2VuZXJhdGlvbiwgVDVUb2tlbml6ZXIKICAgICAgICAKICAgICAgICBzZWxmLmRldmljZSA9IGRldmljZQogICAgICAgIHNlbGYudG9rZW5pemVyID0gVDVUb2tlbml6ZXIuZnJvbV9wcmV0cmFpbmVkKG1vZGVsX25hbWUpCiAgICAgICAgc2VsZi5sbG0gPSBUNUZvckNvbmRpdGlvbmFsR2VuZXJhdGlvbi5mcm9tX3ByZXRyYWluZWQobW9kZWxfbmFtZSkKICAgICAgICAKICAgICAgICBpZiBzZWxmLmRldmljZToKICAgICAgICAgICAgc2VsZi5sbG0gPSBzZWxmLmxsbS50byhzZWxmLmRldmljZSkKICAgICAgICAgICAgCiAgICAgICAgc2VsZi5tb2RlbF9uYW1lID0gbW9kZWxfbmFtZQogICAgICAgIAogICAgICAgIGRldmljZV9pbmZvID0gZiIgb24ge3NlbGYuZGV2aWNlfSIgaWYgc2VsZi5kZXZpY2UgZWxzZSAiIgogICAgICAgIHByaW50KGYi4pyFIExvYWRlZCBMTE06IHttb2RlbF9uYW1lfXtkZXZpY2VfaW5mb30iKQogICAgCiAgICBkZWYgZ2VuZXJhdGUoc2VsZiwgcHJvbXB0LCBtYXhfbmV3X3Rva2Vucz0xMDAsIG51bV9iZWFtcz00KToKICAgICAgICAiIiIKICAgICAgICBHZW5lcmF0ZSB0ZXh0IGZyb20gYSBwcm9tcHQuCiAgICAgICAgCiAgICAgICAgQXJnczoKICAgICAgICAgICAgcHJvbXB0OiBJbnB1dCBwcm9tcHQKICAgICAgICAgICAgbWF4X25ld190b2tlbnM6IE1heGltdW0gdG9rZW5zIHRvIGdlbmVyYXRlCiAgICAgICAgICAgIG51bV9iZWFtczogQmVhbSBzZWFyY2ggd2lkdGgKICAgICAgICAKICAgICAgICBSZXR1cm5zOgogICAgICAgICAgICBzdHI6IEdlbmVyYXRlZCB0ZXh0CiAgICAgICAgIiIiCiAgICAgICAgaW5wdXRzID0gc2VsZi50b2tlbml6ZXIoCiAgICAgICAgICAgIHByb21wdCwKICAgICAgICAgICAgcmV0dXJuX3RlbnNvcnM9InB0IiwKICAgICAgICAgICAgdHJ1bmNhdGlvbj1UcnVlLAogICAgICAgICAgICBtYXhfbGVuZ3RoPTc2OAogICAgICAgICkKICAgICAgICAKICAgICAgICAjIE1vdmUgaW5wdXRzIHRvIHRoZSBzYW1lIGRldmljZSBhcyB0aGUgbW9kZWwKICAgICAgICBpZiBzZWxmLmRldmljZToKICAgICAgICAgICAgaW5wdXRzID0ge2s6IHYudG8oc2VsZi5kZXZpY2UpIGZvciBrLCB2IGluIGlucHV0cy5pdGVtcygpfQogICAgICAgIAogICAgICAgIGlmIHN0cihzZWxmLmRldmljZSkuc3RhcnRzd2l0aCgneGxhJyk6CiAgICAgICAgICAgICMgRm9yY2UgZ3JlZWR5IHNlYXJjaCBvbiBUUFUgdG8gYXZvaWQgYmVhbV9zZWFyY2ggaGFuZ3MgKHRvcmNoLmlzaW4gaXNzdWVzKQogICAgICAgICAgICBudW1fYmVhbXMgPSAxCiAgICAgICAgICAgIAogICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICBvdXRwdXRzID0gc2VsZi5sbG0uZ2VuZXJhdGUoCiAgICAgICAgICAgICAgICAqKmlucHV0cywKICAgICAgICAgICAgICAgIG1heF9uZXdfdG9rZW5zPW1heF9uZXdfdG9rZW5zLAogICAgICAgICAgICAgICAgbnVtX2JlYW1zPW51bV9iZWFtcywKICAgICAgICAgICAgICAgIGVhcmx5X3N0b3BwaW5nPShudW1fYmVhbXMgPiAxKSwKICAgICAgICAgICAgICAgIGRvX3NhbXBsZT1GYWxzZSwKICAgICAgICAgICAgICAgIG5vX3JlcGVhdF9uZ3JhbV9zaXplPTIgaWYgbnVtX2JlYW1zID4gMSBlbHNlIDAKICAgICAgICAgICAgKQogICAgICAgIAogICAgICAgIGFuc3dlciA9IHNlbGYudG9rZW5pemVyLmRlY29kZShvdXRwdXRzWzBdLCBza2lwX3NwZWNpYWxfdG9rZW5zPVRydWUpCiAgICAgICAgcmV0dXJuIGFuc3dlcgogICAgCiAgICBkZWYgZ2VuZXJhdGVfcWEoc2VsZiwgcXVlc3Rpb24sIGNvbnRleHQsIG1heF9uZXdfdG9rZW5zPTEwMCk6CiAgICAgICAgIiIiCiAgICAgICAgR2VuZXJhdGUgYW5zd2VyIGZvciBhIHF1ZXN0aW9uIGdpdmVuIGNvbnRleHQuCiAgICAgICAgCiAgICAgICAgQXJnczoKICAgICAgICAgICAgcXVlc3Rpb246IFRoZSBxdWVzdGlvbgogICAgICAgICAgICBjb250ZXh0OiBDb250ZXh0L3Bhc3NhZ2UgdG8gYW5zd2VyIGZyb20KICAgICAgICAgICAgbWF4X25ld190b2tlbnM6IE1heGltdW0gdG9rZW5zIHRvIGdlbmVyYXRlCiAgICAgICAgCiAgICAgICAgUmV0dXJuczoKICAgICAgICAgICAgdHVwbGU6IChhbnN3ZXIsIHByb21wdCkKICAgICAgICAiIiIKICAgICAgICBwcm9tcHQgPSBmIiIiQW5zd2VyIHRoZSBxdWVzdGlvbiBiYXNlZCBvbiB0aGUgY29udGV4dCBwcm92aWRlZC4KRXh0cmFjdCBzcGVjaWZpYyBkZXRhaWxzOiBleGFjdCBudW1iZXJzLCBuYW1lcywgZGF0ZXMsIGFuZCB0ZWNobmljYWwgdGVybXMuClByb3ZpZGUgdGhlIG1vc3QgcmVsZXZhbnQgYW5zd2VyIGZyb20gdGhlIGNvbnRleHQsIGV2ZW4gaWYgcGFydGlhbC4KCkNvbnRleHQ6Cntjb250ZXh0fQoKUXVlc3Rpb246IHtxdWVzdGlvbn0KCkFuc3dlcjoiIiIKICAgICAgICAKICAgICAgICBhbnN3ZXIgPSBzZWxmLmdlbmVyYXRlKHByb21wdCwgbWF4X25ld190b2tlbnMpCiAgICAgICAgcmV0dXJuIGFuc3dlciwgcHJvbXB0Cg==',
    'projects/hybrid_titans/common/text_utils.py': 'IiIiClRleHQgdXRpbGl0aWVzIGZvciBkb2N1bWVudCBwcm9jZXNzaW5nLgpTaGFyZWQgY2h1bmtpbmcgYW5kIHRleHQgcHJvY2Vzc2luZyBmdW5jdGlvbnMuCiIiIgoKaW1wb3J0IHJlCgoKZGVmIHNwbGl0X2ludG9fY2h1bmtzKHRleHQsIG1pbl9sZW5ndGg9MzAsIHNlbnRlbmNlc19wZXJfY2h1bms9Mywgb3ZlcmxhcF9zZW50ZW5jZXM9MSk6CiAgICAiIiIKICAgIFNwbGl0IHRleHQgaW50byBzZW1hbnRpYyBjaHVua3MgZm9yIHJldHJpZXZhbCB3aXRoIG92ZXJsYXBwaW5nIHdpbmRvd3MuCiAgICAKICAgIFN0cmF0ZWd5OgogICAgLSBIZWFkZXJzIGFyZSBncm91cGVkIHdpdGggdGhlaXIgZm9sbG93aW5nIHNlbnRlbmNlcwogICAgLSBDaHVua3Mgb3ZlcmxhcCBieSAxIHNlbnRlbmNlIHRvIGF2b2lkIGJvdW5kYXJ5IGluZm9ybWF0aW9uIGxvc3MKICAgIC0gRWFjaCBjaHVuayBjb250YWlucyAzIHNlbnRlbmNlcyBmb3IgYmV0dGVyIGNvbnRleHQKICAgIAogICAgQXJnczoKICAgICAgICB0ZXh0OiBJbnB1dCB0ZXh0IChjYW4gY29udGFpbiBtYXJrZG93biBoZWFkZXJzKQogICAgICAgIG1pbl9sZW5ndGg6IE1pbmltdW0gY2h1bmsgbGVuZ3RoIHRvIGtlZXAKICAgICAgICBzZW50ZW5jZXNfcGVyX2NodW5rOiBOdW1iZXIgb2Ygc2VudGVuY2VzIHBlciBjaHVuayAoZGVmYXVsdDogMykKICAgICAgICBvdmVybGFwX3NlbnRlbmNlczogTnVtYmVyIG9mIHNlbnRlbmNlcyB0byBvdmVybGFwIGJldHdlZW4gY2h1bmtzIChkZWZhdWx0OiAxKQogICAgICAgIAogICAgUmV0dXJuczoKICAgICAgICBMaXN0IG9mIHRleHQgY2h1bmtzCiAgICAiIiIKICAgIGxpbmVzID0gdGV4dC5zcGxpdCgnXG4nKQogICAgY2h1bmtzID0gW10KICAgIGN1cnJlbnRfaGVhZGVyID0gIiIKICAgIGFsbF9zZW50ZW5jZXMgPSBbXSAgIyBDb2xsZWN0IGFsbCBzZW50ZW5jZXMgZmlyc3QKICAgIAogICAgIyBGaXJzdCBwYXNzOiBjb2xsZWN0IGFsbCBzZW50ZW5jZXMgd2l0aCB0aGVpciBoZWFkZXJzCiAgICBmb3IgbGluZSBpbiBsaW5lczoKICAgICAgICBsaW5lID0gbGluZS5zdHJpcCgpCiAgICAgICAgaWYgbm90IGxpbmU6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgCiAgICAgICAgIyBDaGVjayBpZiBpdCdzIGEgaGVhZGVyCiAgICAgICAgaWYgbGluZS5zdGFydHN3aXRoKCcjJyk6CiAgICAgICAgICAgICMgSWYgd2UgaGF2ZSBhY2N1bXVsYXRlZCBzZW50ZW5jZXMsIGNyZWF0ZSBjaHVuayBiZWZvcmUgbmV3IGhlYWRlcgogICAgICAgICAgICBpZiBhbGxfc2VudGVuY2VzOgogICAgICAgICAgICAgICAgX2NyZWF0ZV9jaHVua3NfZnJvbV9zZW50ZW5jZXMoCiAgICAgICAgICAgICAgICAgICAgY2h1bmtzLCBjdXJyZW50X2hlYWRlciwgYWxsX3NlbnRlbmNlcywgCiAgICAgICAgICAgICAgICAgICAgc2VudGVuY2VzX3Blcl9jaHVuaywgb3ZlcmxhcF9zZW50ZW5jZXMsIG1pbl9sZW5ndGgKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgICAgIGFsbF9zZW50ZW5jZXMgPSBbXQogICAgICAgICAgICBjdXJyZW50X2hlYWRlciA9IGxpbmUKICAgICAgICAgICAgY29udGludWUKICAgICAgICAKICAgICAgICAjIEhhbmRsZSByZWd1bGFyIGNvbnRlbnQgLSBzcGxpdCBpbnRvIHNlbnRlbmNlcwogICAgICAgICMgUHJvdGVjdCBhYmJyZXZpYXRpb25zCiAgICAgICAgcHJvdGVjdGVkID0gbGluZS5yZXBsYWNlKCdEci4nLCAnRHLCpycpLnJlcGxhY2UoJ01yLicsICdNcsKnJykucmVwbGFjZSgnTXMuJywgJ01zwqcnKQogICAgICAgIHByb3RlY3RlZCA9IHByb3RlY3RlZC5yZXBsYWNlKCdldGMuJywgJ2V0Y8KnJykucmVwbGFjZSgnZS5nLicsICdlZ8KnJykucmVwbGFjZSgnaS5lLicsICdpZcKnJykKICAgICAgICAKICAgICAgICAjIFNwbGl0IG9uIHNlbnRlbmNlIGJvdW5kYXJpZXMKICAgICAgICBwYXJ0cyA9IHJlLnNwbGl0KHInKD88PVsuIT9dKVxzKycsIHByb3RlY3RlZCkKICAgICAgICAKICAgICAgICBmb3IgcGFydCBpbiBwYXJ0czoKICAgICAgICAgICAgcGFydCA9IHBhcnQuc3RyaXAoKQogICAgICAgICAgICAjIFJlc3RvcmUgYWJicmV2aWF0aW9ucwogICAgICAgICAgICBwYXJ0ID0gcGFydC5yZXBsYWNlKCdEcsKnJywgJ0RyLicpLnJlcGxhY2UoJ01ywqcnLCAnTXIuJykucmVwbGFjZSgnTXPCpycsICdNcy4nKQogICAgICAgICAgICBwYXJ0ID0gcGFydC5yZXBsYWNlKCdldGPCpycsICdldGMuJykucmVwbGFjZSgnZWfCpycsICdlLmcuJykucmVwbGFjZSgnaWXCpycsICdpLmUuJykKICAgICAgICAgICAgCiAgICAgICAgICAgIGlmIGxlbihwYXJ0KSA+PSAxMDogICMgTWluaW11bSBzZW50ZW5jZSBsZW5ndGgKICAgICAgICAgICAgICAgIGFsbF9zZW50ZW5jZXMuYXBwZW5kKHBhcnQpCiAgICAKICAgICMgRmx1c2ggcmVtYWluaW5nIHNlbnRlbmNlcwogICAgaWYgYWxsX3NlbnRlbmNlcyBvciBjdXJyZW50X2hlYWRlcjoKICAgICAgICBfY3JlYXRlX2NodW5rc19mcm9tX3NlbnRlbmNlcygKICAgICAgICAgICAgY2h1bmtzLCBjdXJyZW50X2hlYWRlciwgYWxsX3NlbnRlbmNlcywgCiAgICAgICAgICAgIHNlbnRlbmNlc19wZXJfY2h1bmssIG92ZXJsYXBfc2VudGVuY2VzLCBtaW5fbGVuZ3RoCiAgICAgICAgKQogICAgCiAgICByZXR1cm4gY2h1bmtzCgoKZGVmIF9jcmVhdGVfY2h1bmtzX2Zyb21fc2VudGVuY2VzKGNodW5rcywgaGVhZGVyLCBzZW50ZW5jZXMsIAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHNlbnRlbmNlc19wZXJfY2h1bmssIG92ZXJsYXBfc2VudGVuY2VzLCBtaW5fbGVuZ3RoKToKICAgICIiIkhlbHBlcjogQ3JlYXRlIG92ZXJsYXBwaW5nIGNodW5rcyBmcm9tIGEgbGlzdCBvZiBzZW50ZW5jZXMuIiIiCiAgICBpZiBub3Qgc2VudGVuY2VzOgogICAgICAgIGlmIGhlYWRlciBhbmQgbGVuKGhlYWRlcikgPiA1OgogICAgICAgICAgICBjaHVua3MuYXBwZW5kKGhlYWRlcikKICAgICAgICByZXR1cm4KICAgIAogICAgIyBJZiB3ZSBoYXZlIGZld2VyIHNlbnRlbmNlcyB0aGFuIGNodW5rIHNpemUsIGNyZWF0ZSBvbmUgY2h1bmsKICAgIGlmIGxlbihzZW50ZW5jZXMpIDw9IHNlbnRlbmNlc19wZXJfY2h1bms6CiAgICAgICAgY29udGVudCA9ICIgIi5qb2luKHNlbnRlbmNlcykKICAgICAgICBpZiBoZWFkZXI6CiAgICAgICAgICAgIGNodW5rID0gZiJ7aGVhZGVyfSB7Y29udGVudH0iCiAgICAgICAgZWxzZToKICAgICAgICAgICAgY2h1bmsgPSBjb250ZW50CiAgICAgICAgaWYgbGVuKGNodW5rKSA+PSBtaW5fbGVuZ3RoOgogICAgICAgICAgICBjaHVua3MuYXBwZW5kKGNodW5rKQogICAgICAgIHJldHVybgogICAgCiAgICAjIENyZWF0ZSBvdmVybGFwcGluZyBjaHVua3MgdXNpbmcgc2xpZGluZyB3aW5kb3cKICAgIHN0ZXAgPSBzZW50ZW5jZXNfcGVyX2NodW5rIC0gb3ZlcmxhcF9zZW50ZW5jZXMKICAgIHN0ZXAgPSBtYXgoMSwgc3RlcCkgICMgRW5zdXJlIGF0IGxlYXN0IDEgc3RlcAogICAgCiAgICBmb3IgaSBpbiByYW5nZSgwLCBsZW4oc2VudGVuY2VzKSwgc3RlcCk6CiAgICAgICAgd2luZG93ID0gc2VudGVuY2VzW2k6aSArIHNlbnRlbmNlc19wZXJfY2h1bmtdCiAgICAgICAgaWYgbm90IHdpbmRvdzoKICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgCiAgICAgICAgY29udGVudCA9ICIgIi5qb2luKHdpbmRvdykKICAgICAgICAKICAgICAgICAjIEluY2x1ZGUgaGVhZGVyIG9ubHkgaW4gZmlyc3QgY2h1bmsgb2Ygc2VjdGlvbgogICAgICAgIGlmIGkgPT0gMCBhbmQgaGVhZGVyOgogICAgICAgICAgICBjaHVuayA9IGYie2hlYWRlcn0ge2NvbnRlbnR9IgogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGNodW5rID0gY29udGVudAogICAgICAgICAgICAKICAgICAgICBpZiBsZW4oY2h1bmspID49IG1pbl9sZW5ndGg6CiAgICAgICAgICAgIGNodW5rcy5hcHBlbmQoY2h1bmspCiAgICAgICAgCiAgICAgICAgIyBTdG9wIGlmIHdlJ3ZlIHByb2Nlc3NlZCBhbGwgc2VudGVuY2VzCiAgICAgICAgaWYgaSArIHNlbnRlbmNlc19wZXJfY2h1bmsgPj0gbGVuKHNlbnRlbmNlcyk6CiAgICAgICAgICAgIGJyZWFrCgoKZGVmIHNpbXBsZV9zcGxpdCh0ZXh0KToKICAgICIiIlNpbXBsZSBsaW5lLWJ5LWxpbmUgc3BsaXQgKGZvciBjb21wYXJpc29uKS4iIiIKICAgIHJldHVybiBbbGluZS5zdHJpcCgpIGZvciBsaW5lIGluIHRleHQuc3BsaXQoJ1xuJykgaWYgbGluZS5zdHJpcCgpIGFuZCBsZW4obGluZS5zdHJpcCgpKSA+IDEwXQo=',
    'projects/hybrid_titans/baselines.py': 'IiIiCkJhc2VsaW5lIFJldHJpZXZhbCBTdHJhdGVnaWVzIGZvciBUaXRhblJBRyBldmFsdWF0aW9uLgoKVGhyZWUgbW9kZXM6Ci0gUHVyZVJBRzogQk0yNSArIEVtYmVkZGluZyAobm8gTWVtb3J5KQotIFRpdGFuT25seTogTWVtb3J5LWd1aWRlZCByZXRyaWV2YWwgb25seQotIEh5YnJpZFJBRzogQk0yNSArIE1lbW9yeSArIEVtYmVkZGluZyBmdXNpb24KIiIiCgppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRgpmcm9tIHR5cGluZyBpbXBvcnQgTGlzdCwgVHVwbGUsIERpY3QsIE9wdGlvbmFsCgoKZGVmIG5vcm1hbGl6ZV9zY29yZXMoc2NvcmVzOiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoKICAgICIiIk5vcm1hbGl6ZSBzY29yZXMgdG8gWzAsIDFdIHJhbmdlLiIiIgogICAgbWluX3MsIG1heF9zID0gc2NvcmVzLm1pbigpLCBzY29yZXMubWF4KCkKICAgIGlmIG1heF9zIC0gbWluX3MgPCAxZS02OgogICAgICAgIHJldHVybiB0b3JjaC56ZXJvc19saWtlKHNjb3JlcykKICAgIHJldHVybiAoc2NvcmVzIC0gbWluX3MpIC8gKG1heF9zIC0gbWluX3MpCgoKZGVmIGNvbXB1dGVfYm0yNV9zY29yZXMocXVlc3Rpb246IHN0ciwgZG9jdW1lbnRzOiBMaXN0W3N0cl0pIC0+IHRvcmNoLlRlbnNvcjoKICAgICIiIgogICAgQ29tcHV0ZSBzaW1wbGUgQk0yNS1saWtlIGtleXdvcmQgbWF0Y2hpbmcgc2NvcmVzLgogICAgCiAgICBBcmdzOgogICAgICAgIHF1ZXN0aW9uOiBRdWVyeSB0ZXh0CiAgICAgICAgZG9jdW1lbnRzOiBMaXN0IG9mIGRvY3VtZW50IHN0cmluZ3MKICAgIAogICAgUmV0dXJuczoKICAgICAgICB0b3JjaC5UZW5zb3I6IEtleXdvcmQgbWF0Y2ggc2NvcmVzIGZvciBlYWNoIGRvY3VtZW50CiAgICAiIiIKICAgIHF1ZXN0aW9uX2xvd2VyID0gcXVlc3Rpb24ubG93ZXIoKQogICAgcXVlc3Rpb25fd29yZHMgPSBbdyBmb3IgdyBpbiBxdWVzdGlvbl9sb3dlci5zcGxpdCgpIGlmIGxlbih3KSA+IDNdCiAgICAKICAgIHNjb3JlcyA9IFtdCiAgICBmb3IgZG9jIGluIGRvY3VtZW50czoKICAgICAgICBkb2NfbG93ZXIgPSBkb2MubG93ZXIoKQogICAgICAgIHNjb3JlID0gc3VtKDEgZm9yIHdvcmQgaW4gcXVlc3Rpb25fd29yZHMgaWYgd29yZCBpbiBkb2NfbG93ZXIpCiAgICAgICAgc2NvcmVzLmFwcGVuZChmbG9hdChzY29yZSkpCiAgICAKICAgIHJldHVybiB0b3JjaC50ZW5zb3Ioc2NvcmVzKQoKCmNsYXNzIEJhc2VSZXRyaWV2ZXI6CiAgICAiIiJCYXNlIGNsYXNzIGZvciByZXRyaWV2YWwgc3RyYXRlZ2llcy4iIiIKICAgIAogICAgZGVmIF9faW5pdF9fKHNlbGYsIGVtYmVkZGVyLCBsbG1fZ2VuZXJhdG9yKToKICAgICAgICBzZWxmLmVtYmVkZGVyID0gZW1iZWRkZXIKICAgICAgICBzZWxmLmxsbSA9IGxsbV9nZW5lcmF0b3IKICAgIAogICAgZGVmIHJldHJpZXZlKHNlbGYsIHF1ZXN0aW9uOiBzdHIsIGRvY3VtZW50czogTGlzdFtzdHJdLCAKICAgICAgICAgICAgICAgICBkb2NfZW1iZWRkaW5nczogdG9yY2guVGVuc29yLCB0b3BrOiBpbnQgPSA1KSAtPiBUdXBsZVtzdHIsIERpY3RdOgogICAgICAgICIiIgogICAgICAgIFJldHJpZXZlIHJlbGV2YW50IGNvbnRleHQgZnJvbSBkb2N1bWVudHMuCiAgICAgICAgCiAgICAgICAgQXJnczoKICAgICAgICAgICAgcXVlc3Rpb246IFF1ZXJ5IHRleHQKICAgICAgICAgICAgZG9jdW1lbnRzOiBMaXN0IG9mIGRvY3VtZW50IHN0cmluZ3MKICAgICAgICAgICAgZG9jX2VtYmVkZGluZ3M6IFByZS1jb21wdXRlZCBkb2N1bWVudCBlbWJlZGRpbmdzCiAgICAgICAgICAgIHRvcGs6IE51bWJlciBvZiBkb2N1bWVudHMgdG8gcmV0cmlldmUKICAgICAgICAKICAgICAgICBSZXR1cm5zOgogICAgICAgICAgICB0dXBsZTogKGNvbnRleHRfc3RyaW5nLCByZXRyaWV2YWxfZGV0YWlscykKICAgICAgICAiIiIKICAgICAgICByYWlzZSBOb3RJbXBsZW1lbnRlZEVycm9yCiAgICAKICAgIGRlZiBhbnN3ZXIoc2VsZiwgcXVlc3Rpb246IHN0ciwgZG9jdW1lbnRzOiBMaXN0W3N0cl0sCiAgICAgICAgICAgICAgIGRvY19lbWJlZGRpbmdzOiB0b3JjaC5UZW5zb3IsIHRvcGs6IGludCA9IDUsCiAgICAgICAgICAgICAgIG1heF9uZXdfdG9rZW5zOiBpbnQgPSAxMDApIC0+IERpY3Q6CiAgICAgICAgIiIiCiAgICAgICAgRnVsbCBwaXBlbGluZTogcmV0cmlldmUgKyBnZW5lcmF0ZSBhbnN3ZXIuCiAgICAgICAgCiAgICAgICAgUmV0dXJuczoKICAgICAgICAgICAgZGljdDogeyJhbnN3ZXIiOiBzdHIsICJjb250ZXh0Ijogc3RyLCAicmV0cmlldmFsX2RldGFpbHMiOiBkaWN0fQogICAgICAgICIiIgogICAgICAgIGNvbnRleHQsIGRldGFpbHMgPSBzZWxmLnJldHJpZXZlKHF1ZXN0aW9uLCBkb2N1bWVudHMsIGRvY19lbWJlZGRpbmdzLCB0b3BrKQogICAgICAgIGFuc3dlciwgcHJvbXB0ID0gc2VsZi5sbG0uZ2VuZXJhdGVfcWEocXVlc3Rpb24sIGNvbnRleHQsIG1heF9uZXdfdG9rZW5zKQogICAgICAgIAogICAgICAgIHJldHVybiB7CiAgICAgICAgICAgICJhbnN3ZXIiOiBhbnN3ZXIsCiAgICAgICAgICAgICJjb250ZXh0IjogY29udGV4dCwKICAgICAgICAgICAgInByb21wdCI6IHByb21wdCwKICAgICAgICAgICAgInJldHJpZXZhbF9kZXRhaWxzIjogZGV0YWlscywKICAgICAgICAgICAgIm1vZGUiOiBzZWxmLl9fY2xhc3NfXy5fX25hbWVfXwogICAgICAgIH0KCgpjbGFzcyBQdXJlUkFHKEJhc2VSZXRyaWV2ZXIpOgogICAgIiIiCiAgICBQdXJlIFJBRzogQk0yNSArIEVtYmVkZGluZyByZXRyaWV2YWwgKG5vIE1lbW9yeSkuCiAgICAKICAgIFRoaXMgaXMgdGhlIHRyYWRpdGlvbmFsIFJBRyBiYXNlbGluZSB3aXRob3V0IGFueSBUaXRhbiBtZW1vcnkuCiAgICAiIiIKICAgIAogICAgZGVmIF9faW5pdF9fKHNlbGYsIGVtYmVkZGVyLCBsbG1fZ2VuZXJhdG9yLCAKICAgICAgICAgICAgICAgICBibTI1X3dlaWdodDogZmxvYXQgPSAwLjQsIGVtYmVkX3dlaWdodDogZmxvYXQgPSAwLjYpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oZW1iZWRkZXIsIGxsbV9nZW5lcmF0b3IpCiAgICAgICAgc2VsZi5ibTI1X3dlaWdodCA9IGJtMjVfd2VpZ2h0CiAgICAgICAgc2VsZi5lbWJlZF93ZWlnaHQgPSBlbWJlZF93ZWlnaHQKICAgIAogICAgZGVmIHJldHJpZXZlKHNlbGYsIHF1ZXN0aW9uOiBzdHIsIGRvY3VtZW50czogTGlzdFtzdHJdLAogICAgICAgICAgICAgICAgIGRvY19lbWJlZGRpbmdzOiB0b3JjaC5UZW5zb3IsIHRvcGs6IGludCA9IDUpIC0+IFR1cGxlW3N0ciwgRGljdF06CiAgICAgICAgIiIiUmV0cmlldmUgdXNpbmcgQk0yNSArIEVtYmVkZGluZyBvbmx5LiIiIgogICAgICAgIAogICAgICAgICMgRW1iZWQgcXVlc3Rpb24KICAgICAgICBxdWVyeV9lbWIgPSBzZWxmLmVtYmVkZGVyKHF1ZXN0aW9uKQogICAgICAgIHF1ZXJ5X3ZlYyA9IHF1ZXJ5X2VtYi5yZXNoYXBlKC0xLCBzZWxmLmVtYmVkZGVyLnRhcmdldF9kaW0pLm1lYW4oZGltPTApCiAgICAgICAgCiAgICAgICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgICAgICMgQk0yNSBzY29yZXMKICAgICAgICAgICAgYm0yNV9zY29yZXMgPSBjb21wdXRlX2JtMjVfc2NvcmVzKHF1ZXN0aW9uLCBkb2N1bWVudHMpCiAgICAgICAgICAgIAogICAgICAgICAgICAjIEVtYmVkZGluZyBzaW1pbGFyaXR5CiAgICAgICAgICAgIGVtYmVkX3Njb3JlcyA9IEYuY29zaW5lX3NpbWlsYXJpdHkoCiAgICAgICAgICAgICAgICBxdWVyeV92ZWMudW5zcXVlZXplKDApLCBkb2NfZW1iZWRkaW5ncwogICAgICAgICAgICApCiAgICAgICAgICAgIAogICAgICAgICAgICAjIEZ1c2lvbiAobm8gTWVtb3J5ISkKICAgICAgICAgICAgIyBNb3ZlIEJNMjUgc2NvcmVzIHRvIHNhbWUgZGV2aWNlIGFzIGVtYmVkZGluZ3MKICAgICAgICAgICAgYm0yNV9zY29yZXMgPSBibTI1X3Njb3Jlcy50byhkb2NfZW1iZWRkaW5ncy5kZXZpY2UpCiAgICAgICAgICAgIAogICAgICAgICAgICBmdXNlZF9zY29yZXMgPSAoCiAgICAgICAgICAgICAgICBzZWxmLmJtMjVfd2VpZ2h0ICogbm9ybWFsaXplX3Njb3JlcyhibTI1X3Njb3JlcykgKwogICAgICAgICAgICAgICAgc2VsZi5lbWJlZF93ZWlnaHQgKiBub3JtYWxpemVfc2NvcmVzKGVtYmVkX3Njb3JlcykKICAgICAgICAgICAgKQogICAgICAgICAgICAKICAgICAgICAgICAgIyBUb3AtSwogICAgICAgICAgICB0b3BrX3ZhbHVlcywgdG9wa19pbmRpY2VzID0gZnVzZWRfc2NvcmVzLnRvcGsobWluKHRvcGssIGxlbihkb2N1bWVudHMpKSkKICAgICAgICAKICAgICAgICAjIEJ1aWxkIGNvbnRleHQKICAgICAgICBjb250ZXh0X2xpbmVzID0gW2RvY3VtZW50c1tpZHhdIGZvciBpZHggaW4gdG9wa19pbmRpY2VzLnRvbGlzdCgpXQogICAgICAgIGNvbnRleHQgPSAiICIuam9pbihjb250ZXh0X2xpbmVzKQogICAgICAgIAogICAgICAgIHJldHVybiBjb250ZXh0LCB7CiAgICAgICAgICAgICJtb2RlIjogIlB1cmVSQUciLAogICAgICAgICAgICAidG9wa19pbmRpY2VzIjogdG9wa19pbmRpY2VzLnRvbGlzdCgpLAogICAgICAgICAgICAiYm0yNV90b3AxIjogYm0yNV9zY29yZXMuYXJnbWF4KCkuaXRlbSgpLAogICAgICAgICAgICAiZW1iZWRfdG9wMSI6IGVtYmVkX3Njb3Jlcy5hcmdtYXgoKS5pdGVtKCksCiAgICAgICAgICAgICJmdXNlZF90b3AxIjogZnVzZWRfc2NvcmVzLmFyZ21heCgpLml0ZW0oKQogICAgICAgIH0KCgpjbGFzcyBUaXRhbk9ubHkoQmFzZVJldHJpZXZlcik6CiAgICAiIiIKICAgIFRpdGFuLU9ubHk6IE1lbW9yeS1ndWlkZWQgcmV0cmlldmFsLgogICAgCiAgICBVc2VzIG9ubHkgdGhlIFRpdGFuIG1lbW9yeSBvdXRwdXQgZm9yIHJldHJpZXZhbCBzY29yaW5nLgogICAgTm8gQk0yNSwgbm8gZGlyZWN0IGVtYmVkZGluZyBzaW1pbGFyaXR5LgogICAgIiIiCiAgICAKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBlbWJlZGRlciwgbGxtX2dlbmVyYXRvciwgdGl0YW5fcmFnKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKGVtYmVkZGVyLCBsbG1fZ2VuZXJhdG9yKQogICAgICAgIHNlbGYudGl0YW5fcmFnID0gdGl0YW5fcmFnCiAgICAKICAgIGRlZiByZXRyaWV2ZShzZWxmLCBxdWVzdGlvbjogc3RyLCBkb2N1bWVudHM6IExpc3Rbc3RyXSwKICAgICAgICAgICAgICAgICBkb2NfZW1iZWRkaW5nczogdG9yY2guVGVuc29yLCB0b3BrOiBpbnQgPSA1KSAtPiBUdXBsZVtzdHIsIERpY3RdOgogICAgICAgICIiIlJldHJpZXZlIHVzaW5nIE1lbW9yeSBvdXRwdXQgb25seS4iIiIKICAgICAgICAKICAgICAgICBkaW0gPSBzZWxmLmVtYmVkZGVyLnRhcmdldF9kaW0KICAgICAgICAKICAgICAgICAjIEVtYmVkIHF1ZXN0aW9uCiAgICAgICAgcXVlcnlfZW1iID0gc2VsZi5lbWJlZGRlcihxdWVzdGlvbikKICAgICAgICBxdWVyeV9mbGF0ID0gcXVlcnlfZW1iLnJlc2hhcGUoLTEsIGRpbSkKICAgICAgICAKICAgICAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICAgICAgIyBHZXQgbWVtb3J5IG91dHB1dAogICAgICAgICAgICBtZW1vcnlfb3V0cHV0ID0gc2VsZi50aXRhbl9yYWcudGl0YW4ubHRtLmZvcndhcmRfbm9fdXBkYXRlKHF1ZXJ5X2ZsYXQpCiAgICAgICAgICAgIG1lbW9yeV92ZWMgPSBtZW1vcnlfb3V0cHV0Lm1lYW4oZGltPTApCiAgICAgICAgICAgIAogICAgICAgICAgICAjIE1lbW9yeS1iYXNlZCBzaW1pbGFyaXR5IChvbmx5IHNpZ25hbCEpCiAgICAgICAgICAgIG1lbW9yeV9zY29yZXMgPSBGLmNvc2luZV9zaW1pbGFyaXR5KAogICAgICAgICAgICAgICAgbWVtb3J5X3ZlYy51bnNxdWVlemUoMCksIGRvY19lbWJlZGRpbmdzCiAgICAgICAgICAgICkKICAgICAgICAgICAgCiAgICAgICAgICAgICMgVG9wLUsKICAgICAgICAgICAgdG9wa192YWx1ZXMsIHRvcGtfaW5kaWNlcyA9IG1lbW9yeV9zY29yZXMudG9wayhtaW4odG9waywgbGVuKGRvY3VtZW50cykpKQogICAgICAgIAogICAgICAgICMgQnVpbGQgY29udGV4dAogICAgICAgIGNvbnRleHRfbGluZXMgPSBbZG9jdW1lbnRzW2lkeF0gZm9yIGlkeCBpbiB0b3BrX2luZGljZXMudG9saXN0KCldCiAgICAgICAgY29udGV4dCA9ICIgIi5qb2luKGNvbnRleHRfbGluZXMpCiAgICAgICAgCiAgICAgICAgcmV0dXJuIGNvbnRleHQsIHsKICAgICAgICAgICAgIm1vZGUiOiAiVGl0YW5Pbmx5IiwKICAgICAgICAgICAgInRvcGtfaW5kaWNlcyI6IHRvcGtfaW5kaWNlcy50b2xpc3QoKSwKICAgICAgICAgICAgIm1lbW9yeV90b3AxIjogbWVtb3J5X3Njb3Jlcy5hcmdtYXgoKS5pdGVtKCkKICAgICAgICB9CiAgICAKICAgIGRlZiBvbmxpbmVfbGVhcm4oc2VsZiwgcXVlc3Rpb246IHN0ciwgZXhwZWN0ZWRfYW5zd2VyOiBzdHIpOgogICAgICAgICIiIkxlYXJuIGZyb20gUSZBIHBhaXIgKE9ubGluZSBMZWFybmluZykuIiIiCiAgICAgICAgZGltID0gc2VsZi5lbWJlZGRlci50YXJnZXRfZGltCiAgICAgICAgCiAgICAgICAgcXVlcnlfZW1iID0gc2VsZi5lbWJlZGRlcihxdWVzdGlvbikKICAgICAgICBxdWVyeV92ZWMgPSBxdWVyeV9lbWIucmVzaGFwZSgtMSwgZGltKS5tZWFuKGRpbT0wLCBrZWVwZGltPVRydWUpCiAgICAgICAgCiAgICAgICAgYW5zd2VyX2VtYiA9IHNlbGYuZW1iZWRkZXIoZXhwZWN0ZWRfYW5zd2VyKQogICAgICAgIGFuc3dlcl92ZWMgPSBhbnN3ZXJfZW1iLnJlc2hhcGUoLTEsIGRpbSkubWVhbihkaW09MCwga2VlcGRpbT1UcnVlKQogICAgICAgIAogICAgICAgIHNlbGYudGl0YW5fcmFnLnRpdGFuLmx0bS5mb3J3YXJkX3dpdGhfdXBkYXRlKHF1ZXJ5X3ZlYywgYW5zd2VyX3ZlYykKCgpjbGFzcyBIeWJyaWRSQUcoQmFzZVJldHJpZXZlcik6CiAgICAiIiIKICAgIEh5YnJpZCBSQUc6IEJNMjUgKyBNZW1vcnkgKyBFbWJlZGRpbmcgZnVzaW9uIHdpdGggZHluYW1pYyB3ZWlnaHRpbmcuCiAgICAKICAgIEtleSBpbXByb3ZlbWVudDogTWVtb3J5IHdlaWdodCBpcyBhZGp1c3RlZCBkeW5hbWljYWxseSBiYXNlZCBvbiAKICAgIG1lbW9yeSBjb25maWRlbmNlIChvdXRwdXQgc3RyZW5ndGgpLiBMb3cgY29uZmlkZW5jZSA9IHJlZHVjZSBtZW1vcnkgd2VpZ2h0LgogICAgIiIiCiAgICAKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBlbWJlZGRlciwgbGxtX2dlbmVyYXRvciwgdGl0YW5fcmFnLAogICAgICAgICAgICAgICAgIGJhc2VfYm0yNV93ZWlnaHQ6IGZsb2F0ID0gMC4zLAogICAgICAgICAgICAgICAgIGJhc2VfbWVtb3J5X3dlaWdodDogZmxvYXQgPSAwLjMsCiAgICAgICAgICAgICAgICAgYmFzZV9lbWJlZF93ZWlnaHQ6IGZsb2F0ID0gMC40KToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKGVtYmVkZGVyLCBsbG1fZ2VuZXJhdG9yKQogICAgICAgIHNlbGYudGl0YW5fcmFnID0gdGl0YW5fcmFnCiAgICAgICAgc2VsZi5iYXNlX2JtMjVfd2VpZ2h0ID0gYmFzZV9ibTI1X3dlaWdodAogICAgICAgIHNlbGYuYmFzZV9tZW1vcnlfd2VpZ2h0ID0gYmFzZV9tZW1vcnlfd2VpZ2h0CiAgICAgICAgc2VsZi5iYXNlX2VtYmVkX3dlaWdodCA9IGJhc2VfZW1iZWRfd2VpZ2h0CiAgICAKICAgIGRlZiBfY29tcHV0ZV9tZW1vcnlfY29uZmlkZW5jZShzZWxmLCBxdWVyeV9lbWIsIG1lbW9yeV9vdXRwdXQpOgogICAgICAgICIiIgogICAgICAgIENvbXB1dGUgbWVtb3J5IGNvbmZpZGVuY2UgYmFzZWQgb24gb3V0cHV0IHN0cmVuZ3RoLgogICAgICAgIAogICAgICAgIEhpZ2hlciBjb25maWRlbmNlIHdoZW4gbWVtb3J5IG91dHB1dCBub3JtIGlzIHNpbWlsYXIgdG8gaW5wdXQgbm9ybS4KICAgICAgICBSZXR1cm5zIHZhbHVlIGluIFswLCAxXSB3aGVyZSAxID0gaGlnaCBjb25maWRlbmNlLgogICAgICAgICIiIgogICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICBpbnB1dF9ub3JtID0gdG9yY2gubm9ybShxdWVyeV9lbWIpCiAgICAgICAgICAgIG91dHB1dF9ub3JtID0gdG9yY2gubm9ybShtZW1vcnlfb3V0cHV0KQogICAgICAgICAgICAKICAgICAgICAgICAgIyBSYXRpbyBvZiBvdXRwdXQgdG8gaW5wdXQgbm9ybQogICAgICAgICAgICBjb25maWRlbmNlX3JhdGlvID0gb3V0cHV0X25vcm0gLyAoaW5wdXRfbm9ybSArIDFlLTgpCiAgICAgICAgICAgIAogICAgICAgICAgICAjIE5vcm1hbGl6ZSB0byAwLTEgcmFuZ2UgKGFzc3VtaW5nIHR5cGljYWwgcmF0aW8gaXMgMC0yKQogICAgICAgICAgICBjb25maWRlbmNlID0gdG9yY2guY2xhbXAoY29uZmlkZW5jZV9yYXRpbyAvIDIuMCwgMC4wLCAxLjApLml0ZW0oKQogICAgICAgICAgICAKICAgICAgICAgICAgcmV0dXJuIGNvbmZpZGVuY2UKICAgIAogICAgZGVmIHJldHJpZXZlKHNlbGYsIHF1ZXN0aW9uOiBzdHIsIGRvY3VtZW50czogTGlzdFtzdHJdLAogICAgICAgICAgICAgICAgIGRvY19lbWJlZGRpbmdzOiB0b3JjaC5UZW5zb3IsIHRvcGs6IGludCA9IDUpIC0+IFR1cGxlW3N0ciwgRGljdF06CiAgICAgICAgIiIiUmV0cmlldmUgdXNpbmcgQk0yNSArIE1lbW9yeSArIEVtYmVkZGluZyBmdXNpb24gd2l0aCBkeW5hbWljIHdlaWdodHMuIiIiCiAgICAgICAgCiAgICAgICAgZGltID0gc2VsZi5lbWJlZGRlci50YXJnZXRfZGltCiAgICAgICAgCiAgICAgICAgIyBFbWJlZCBxdWVzdGlvbgogICAgICAgIHF1ZXJ5X2VtYiA9IHNlbGYuZW1iZWRkZXIocXVlc3Rpb24pCiAgICAgICAgcXVlcnlfZmxhdCA9IHF1ZXJ5X2VtYi5yZXNoYXBlKC0xLCBkaW0pCiAgICAgICAgcXVlcnlfdmVjID0gcXVlcnlfZmxhdC5tZWFuKGRpbT0wKQogICAgICAgIAogICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICAjIFNpZ25hbCAxOiBCTTI1CiAgICAgICAgICAgIGJtMjVfc2NvcmVzID0gY29tcHV0ZV9ibTI1X3Njb3JlcyhxdWVzdGlvbiwgZG9jdW1lbnRzKQogICAgICAgICAgICAKICAgICAgICAgICAgIyBTaWduYWwgMjogTWVtb3J5CiAgICAgICAgICAgIG1lbW9yeV9vdXRwdXQgPSBzZWxmLnRpdGFuX3JhZy50aXRhbi5sdG0uZm9yd2FyZF9ub191cGRhdGUocXVlcnlfZmxhdCkKICAgICAgICAgICAgbWVtb3J5X3ZlYyA9IG1lbW9yeV9vdXRwdXQubWVhbihkaW09MCkKICAgICAgICAgICAgbWVtb3J5X3Njb3JlcyA9IEYuY29zaW5lX3NpbWlsYXJpdHkoCiAgICAgICAgICAgICAgICBtZW1vcnlfdmVjLnVuc3F1ZWV6ZSgwKSwgZG9jX2VtYmVkZGluZ3MKICAgICAgICAgICAgKQogICAgICAgICAgICAKICAgICAgICAgICAgIyBTaWduYWwgMzogRGlyZWN0IEVtYmVkZGluZwogICAgICAgICAgICBlbWJlZF9zY29yZXMgPSBGLmNvc2luZV9zaW1pbGFyaXR5KAogICAgICAgICAgICAgICAgcXVlcnlfdmVjLnVuc3F1ZWV6ZSgwKSwgZG9jX2VtYmVkZGluZ3MKICAgICAgICAgICAgKQogICAgICAgICAgICAKICAgICAgICAgICAgIyBDb21wdXRlIG1lbW9yeSBjb25maWRlbmNlCiAgICAgICAgICAgIG1lbW9yeV9jb25maWRlbmNlID0gc2VsZi5fY29tcHV0ZV9tZW1vcnlfY29uZmlkZW5jZShxdWVyeV9mbGF0LCBtZW1vcnlfb3V0cHV0KQogICAgICAgICAgICAKICAgICAgICAgICAgIyBEeW5hbWljIHdlaWdodCBhZGp1c3RtZW50IGJhc2VkIG9uIGNvbmZpZGVuY2UKICAgICAgICAgICAgIyBMb3cgY29uZmlkZW5jZSDihpIgcmVkdWNlIG1lbW9yeSB3ZWlnaHQsIGluY3JlYXNlIGVtYmVkZGluZyB3ZWlnaHQKICAgICAgICAgICAgbWVtb3J5X3dlaWdodCA9IHNlbGYuYmFzZV9tZW1vcnlfd2VpZ2h0ICogbWVtb3J5X2NvbmZpZGVuY2UKICAgICAgICAgICAgZW1iZWRfd2VpZ2h0ID0gc2VsZi5iYXNlX2VtYmVkX3dlaWdodCArIHNlbGYuYmFzZV9tZW1vcnlfd2VpZ2h0ICogKDEgLSBtZW1vcnlfY29uZmlkZW5jZSkKICAgICAgICAgICAgYm0yNV93ZWlnaHQgPSBzZWxmLmJhc2VfYm0yNV93ZWlnaHQKICAgICAgICAgICAgCiAgICAgICAgICAgICMgTm9ybWFsaXplIHdlaWdodHMgdG8gc3VtIHRvIDEKICAgICAgICAgICAgdG90YWwgPSBibTI1X3dlaWdodCArIG1lbW9yeV93ZWlnaHQgKyBlbWJlZF93ZWlnaHQKICAgICAgICAgICAgYm0yNV93ZWlnaHQgLz0gdG90YWwKICAgICAgICAgICAgbWVtb3J5X3dlaWdodCAvPSB0b3RhbAogICAgICAgICAgICBlbWJlZF93ZWlnaHQgLz0gdG90YWwKICAgICAgICAgICAgCiAgICAgICAgICAgICMgRnVzaW9uIHdpdGggZHluYW1pYyB3ZWlnaHRzCiAgICAgICAgICAgICMgTW92ZSBCTTI1IHNjb3JlcyB0byBzYW1lIGRldmljZQogICAgICAgICAgICBibTI1X3Njb3JlcyA9IGJtMjVfc2NvcmVzLnRvKGRvY19lbWJlZGRpbmdzLmRldmljZSkKICAgICAgICAgICAgCiAgICAgICAgICAgIGZ1c2VkX3Njb3JlcyA9ICgKICAgICAgICAgICAgICAgIGJtMjVfd2VpZ2h0ICogbm9ybWFsaXplX3Njb3JlcyhibTI1X3Njb3JlcykgKwogICAgICAgICAgICAgICAgbWVtb3J5X3dlaWdodCAqIG5vcm1hbGl6ZV9zY29yZXMobWVtb3J5X3Njb3JlcykgKwogICAgICAgICAgICAgICAgZW1iZWRfd2VpZ2h0ICogbm9ybWFsaXplX3Njb3JlcyhlbWJlZF9zY29yZXMpCiAgICAgICAgICAgICkKICAgICAgICAgICAgCiAgICAgICAgICAgICMgVG9wLUsKICAgICAgICAgICAgdG9wa192YWx1ZXMsIHRvcGtfaW5kaWNlcyA9IGZ1c2VkX3Njb3Jlcy50b3BrKG1pbih0b3BrLCBsZW4oZG9jdW1lbnRzKSkpCiAgICAgICAgCiAgICAgICAgIyBCdWlsZCBjb250ZXh0CiAgICAgICAgY29udGV4dF9saW5lcyA9IFtkb2N1bWVudHNbaWR4XSBmb3IgaWR4IGluIHRvcGtfaW5kaWNlcy50b2xpc3QoKV0KICAgICAgICBjb250ZXh0ID0gIiAiLmpvaW4oY29udGV4dF9saW5lcykKICAgICAgICAKICAgICAgICByZXR1cm4gY29udGV4dCwgewogICAgICAgICAgICAibW9kZSI6ICJIeWJyaWRSQUciLAogICAgICAgICAgICAidG9wa19pbmRpY2VzIjogdG9wa19pbmRpY2VzLnRvbGlzdCgpLAogICAgICAgICAgICAiYm0yNV90b3AxIjogYm0yNV9zY29yZXMuYXJnbWF4KCkuaXRlbSgpLAogICAgICAgICAgICAibWVtb3J5X3RvcDEiOiBtZW1vcnlfc2NvcmVzLmFyZ21heCgpLml0ZW0oKSwKICAgICAgICAgICAgImVtYmVkX3RvcDEiOiBlbWJlZF9zY29yZXMuYXJnbWF4KCkuaXRlbSgpLAogICAgICAgICAgICAiZnVzZWRfdG9wMSI6IGZ1c2VkX3Njb3Jlcy5hcmdtYXgoKS5pdGVtKCksCiAgICAgICAgICAgICJtZW1vcnlfY29uZmlkZW5jZSI6IG1lbW9yeV9jb25maWRlbmNlLAogICAgICAgICAgICAiZHluYW1pY193ZWlnaHRzIjogewogICAgICAgICAgICAgICAgImJtMjUiOiByb3VuZChibTI1X3dlaWdodCwgMyksCiAgICAgICAgICAgICAgICAibWVtb3J5Ijogcm91bmQobWVtb3J5X3dlaWdodCwgMyksCiAgICAgICAgICAgICAgICAiZW1iZWQiOiByb3VuZChlbWJlZF93ZWlnaHQsIDMpCiAgICAgICAgICAgIH0KICAgICAgICB9CiAgICAKICAgIGRlZiBvbmxpbmVfbGVhcm4oc2VsZiwgcXVlc3Rpb246IHN0ciwgZXhwZWN0ZWRfYW5zd2VyOiBzdHIpOgogICAgICAgICIiIkxlYXJuIGZyb20gUSZBIHBhaXIgKE9ubGluZSBMZWFybmluZykuIiIiCiAgICAgICAgZGltID0gc2VsZi5lbWJlZGRlci50YXJnZXRfZGltCiAgICAgICAgCiAgICAgICAgcXVlcnlfZW1iID0gc2VsZi5lbWJlZGRlcihxdWVzdGlvbikKICAgICAgICBxdWVyeV92ZWMgPSBxdWVyeV9lbWIucmVzaGFwZSgtMSwgZGltKS5tZWFuKGRpbT0wLCBrZWVwZGltPVRydWUpCiAgICAgICAgCiAgICAgICAgYW5zd2VyX2VtYiA9IHNlbGYuZW1iZWRkZXIoZXhwZWN0ZWRfYW5zd2VyKQogICAgICAgIGFuc3dlcl92ZWMgPSBhbnN3ZXJfZW1iLnJlc2hhcGUoLTEsIGRpbSkubWVhbihkaW09MCwga2VlcGRpbT1UcnVlKQogICAgICAgIAogICAgICAgIHNlbGYudGl0YW5fcmFnLnRpdGFuLmx0bS5mb3J3YXJkX3dpdGhfdXBkYXRlKHF1ZXJ5X3ZlYywgYW5zd2VyX3ZlYykKCgpkZWYgY3JlYXRlX3JldHJpZXZlcnMoZW1iZWRkZXIsIGxsbV9nZW5lcmF0b3IsIHRpdGFuX3JhZz1Ob25lKSAtPiBEaWN0W3N0ciwgQmFzZVJldHJpZXZlcl06CiAgICAiIiIKICAgIEZhY3RvcnkgZnVuY3Rpb24gdG8gY3JlYXRlIGFsbCByZXRyaWV2ZXIgaW5zdGFuY2VzLgogICAgCiAgICBBcmdzOgogICAgICAgIGVtYmVkZGVyOiBTZW50ZW5jZVRyYW5zZm9ybWVyRW1iZWRkZXIgaW5zdGFuY2UKICAgICAgICBsbG1fZ2VuZXJhdG9yOiBGbGFuVDVHZW5lcmF0b3IgaW5zdGFuY2UKICAgICAgICB0aXRhbl9yYWc6IFRpdGFuUkFHIGluc3RhbmNlIChyZXF1aXJlZCBmb3IgVGl0YW5Pbmx5IGFuZCBIeWJyaWRSQUcpCiAgICAKICAgIFJldHVybnM6CiAgICAgICAgZGljdDogeyJwdXJlX3JhZyI6IFB1cmVSQUcsICJ0aXRhbl9vbmx5IjogVGl0YW5Pbmx5LCAiaHlicmlkIjogSHlicmlkUkFHfQogICAgIiIiCiAgICByZXRyaWV2ZXJzID0gewogICAgICAgICJwdXJlX3JhZyI6IFB1cmVSQUcoZW1iZWRkZXIsIGxsbV9nZW5lcmF0b3IpCiAgICB9CiAgICAKICAgIGlmIHRpdGFuX3JhZyBpcyBub3QgTm9uZToKICAgICAgICByZXRyaWV2ZXJzWyJ0aXRhbl9vbmx5Il0gPSBUaXRhbk9ubHkoZW1iZWRkZXIsIGxsbV9nZW5lcmF0b3IsIHRpdGFuX3JhZykKICAgICAgICByZXRyaWV2ZXJzWyJoeWJyaWQiXSA9IEh5YnJpZFJBRyhlbWJlZGRlciwgbGxtX2dlbmVyYXRvciwgdGl0YW5fcmFnKQogICAgCiAgICByZXR1cm4gcmV0cmlldmVycwo=',
    'projects/hybrid_titans/compare_baselines.py': 'IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiIKQmFzZWxpbmUgQ29tcGFyaXNvbiBTY3JpcHQgZm9yIFRpdGFuUkFHLgoKQ29tcGFyZXMgdGhyZWUgcmV0cmlldmFsIHN0cmF0ZWdpZXM6Ci0gUHVyZVJBRzogQk0yNSArIEVtYmVkZGluZyAobm8gTWVtb3J5KQotIFRpdGFuT25seTogTWVtb3J5LWd1aWRlZCByZXRyaWV2YWwgb25seSAgCi0gSHlicmlkUkFHOiBCTTI1ICsgTWVtb3J5ICsgRW1iZWRkaW5nIGZ1c2lvbgoKVXNhZ2U6CiAgICAjIFVzZSBzYW1wbGUgZXNzYXlzIChvcmlnaW5hbCBtb2RlKQogICAgcHl0aG9uIGNvbXBhcmVfYmFzZWxpbmVzLnB5IC0tZXNzYXkgY2xpbWF0ZSAtLWVwb2NocyA1MAogICAgCiAgICAjIFVzZSBTUXVBRCBkYXRhc2V0IChuZXcgbW9kZSkKICAgIHB5dGhvbiBjb21wYXJlX2Jhc2VsaW5lcy5weSAtLXNxdWFkIC0tdGl0bGVzIDEwIC0tZXBvY2hzIDUwCiIiIgoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBzeXMKaW1wb3J0IG9zCmltcG9ydCByZQppbXBvcnQgcmFuZG9tCmltcG9ydCBpbwoKIyBGaXggV2luZG93cyBjb25zb2xlIGVuY29kaW5nCmlmIHN5cy5wbGF0Zm9ybSA9PSAnd2luMzInOgogICAgc3lzLnN0ZG91dCA9IGlvLlRleHRJT1dyYXBwZXIoc3lzLnN0ZG91dC5idWZmZXIsIGVuY29kaW5nPSd1dGYtOCcsIGVycm9ycz0ncmVwbGFjZScpCiAgICBzeXMuc3RkZXJyID0gaW8uVGV4dElPV3JhcHBlcihzeXMuc3RkZXJyLmJ1ZmZlciwgZW5jb2Rpbmc9J3V0Zi04JywgZXJyb3JzPSdyZXBsYWNlJykKCiMgQWRkIHBhdGhzCnN5cy5wYXRoLmluc2VydCgwLCBvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmFic3BhdGgoX19maWxlX18pKSkpKQpzeXMucGF0aC5pbnNlcnQoMCwgb3MucGF0aC5qb2luKG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguYWJzcGF0aChfX2ZpbGVfXykpKSksICdzcmMnKSkKCmZyb20gY29tbW9uLmVtYmVkZGVycyBpbXBvcnQgU2VudGVuY2VUcmFuc2Zvcm1lckVtYmVkZGVyCmZyb20gY29tbW9uLnRpdGFuX3V0aWxzIGltcG9ydCBjcmVhdGVfdGl0YW5fcmFnCmZyb20gY29tbW9uLmxsbV91dGlscyBpbXBvcnQgRmxhblQ1R2VuZXJhdG9yCmZyb20gY29tbW9uLnRleHRfdXRpbHMgaW1wb3J0IHNwbGl0X2ludG9fY2h1bmtzCmZyb20gYmFzZWxpbmVzIGltcG9ydCBjcmVhdGVfcmV0cmlldmVycwoKCiMgSW1wb3J0IGVzc2F5cyBmcm9tIGRhdGEgZGlyZWN0b3J5CnN5cy5wYXRoLmluc2VydCgwLCBvcy5wYXRoLmpvaW4ob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5hYnNwYXRoKF9fZmlsZV9fKSkpKSwgJ2RhdGEnKSkKCgpkZWYgbG9hZF9zYW1wbGVfZXNzYXlzKCk6CiAgICAiIiJMb2FkIHNhbXBsZSBlc3NheXMgZm9yIG9yaWdpbmFsIG1vZGUuIiIiCiAgICBmcm9tIHNhbXBsZV9lc3NheXMgaW1wb3J0IGdldF9hbGxfZXNzYXlzLCBnZXRfdGVzdF9xdWVzdGlvbnMKICAgIHJldHVybiBnZXRfYWxsX2Vzc2F5cygpLCBnZXRfdGVzdF9xdWVzdGlvbnMoKQoKCmRlZiBsb2FkX3NxdWFkX2RhdGEoKToKICAgICIiIkxvYWQgcHJvY2Vzc2VkIFNRdUFEIGRhdGEuIiIiCiAgICB0cnk6CiAgICAgICAgZnJvbSBwcm9jZXNzZWRfc3F1YWQgaW1wb3J0IGdldF9hbGxfY29udGV4dHMsIGdldF90ZXN0X3F1ZXN0aW9ucwogICAgICAgIHJldHVybiBnZXRfYWxsX2NvbnRleHRzKCksIGdldF90ZXN0X3F1ZXN0aW9ucygpCiAgICBleGNlcHQgSW1wb3J0RXJyb3I6CiAgICAgICAgcHJpbnQoIuKdjCBFcnJvcjogcHJvY2Vzc2VkX3NxdWFkLnB5IG5vdCBmb3VuZC4gUGxlYXNlIHJ1biBzY3JpcHQucHkgZmlyc3QuIikKICAgICAgICBzeXMuZXhpdCgxKQoKCmRlZiBleHRyYWN0X2tleV9lbGVtZW50cyh0ZXh0KToKICAgICIiIkV4dHJhY3Qga2V5IG51bWJlcnMgYW5kIGtleXdvcmRzIGZyb20gdGV4dCBmb3IgbWF0Y2hpbmcuIiIiCiAgICB0ZXh0ID0gdGV4dC5sb3dlcigpCiAgICBudW1iZXJzID0gcmUuZmluZGFsbChyJ1tcZC5dKyU/JywgdGV4dCkKICAgIHllYXJzID0gcmUuZmluZGFsbChyJ1xiKDE5fDIwKVxkezJ9XGInLCB0ZXh0KQogICAgcmV0dXJuIHNldChudW1iZXJzICsgeWVhcnMpCgoKZGVmIGV4dHJhY3Rfa2V5d29yZHModGV4dCk6CiAgICAiIiJFeHRyYWN0IG1lYW5pbmdmdWwga2V5d29yZHMgZnJvbSB0ZXh0LCByZW1vdmluZyBzdG9wd29yZHMuIiIiCiAgICBzdG9wd29yZHMgPSB7J2EnLCAnYW4nLCAndGhlJywgJ2lzJywgJ2FyZScsICd3YXMnLCAnd2VyZScsICdiZScsICdiZWVuJywgJ2JlaW5nJywKICAgICAgICAgICAgICAgICAnaGF2ZScsICdoYXMnLCAnaGFkJywgJ2RvJywgJ2RvZXMnLCAnZGlkJywgJ3dpbGwnLCAnd291bGQnLCAnY291bGQnLAogICAgICAgICAgICAgICAgICdzaG91bGQnLCAnbWF5JywgJ21pZ2h0JywgJ211c3QnLCAnc2hhbGwnLCAnY2FuJywgJ25lZWQnLCAnZGFyZScsCiAgICAgICAgICAgICAgICAgJ3RvJywgJ29mJywgJ2luJywgJ2ZvcicsICdvbicsICd3aXRoJywgJ2F0JywgJ2J5JywgJ2Zyb20nLCAnYXMnLAogICAgICAgICAgICAgICAgICdpbnRvJywgJ3Rocm91Z2gnLCAnZHVyaW5nJywgJ2JlZm9yZScsICdhZnRlcicsICdhYm92ZScsICdiZWxvdycsCiAgICAgICAgICAgICAgICAgJ2JldHdlZW4nLCAndW5kZXInLCAnYWdhaW4nLCAnZnVydGhlcicsICd0aGVuJywgJ29uY2UnLCAnYW5kJywgJ29yJywKICAgICAgICAgICAgICAgICAnYnV0JywgJ2lmJywgJ2JlY2F1c2UnLCAndW50aWwnLCAnd2hpbGUnLCAndGhhdCcsICd3aGljaCcsICd3aG8nLAogICAgICAgICAgICAgICAgICd3aG9tJywgJ3RoaXMnLCAndGhlc2UnLCAndGhvc2UnLCAnaXQnLCAnaXRzJywgJ3RoZWlyJywgJ3RoZXknfQogICAgCiAgICB0ZXh0ID0gcmUuc3ViKHInW15cd1xzXScsICcgJywgdGV4dC5sb3dlcigpKQogICAgd29yZHMgPSB0ZXh0LnNwbGl0KCkKICAgIGtleXdvcmRzID0gW3cgZm9yIHcgaW4gd29yZHMgaWYgdyBub3QgaW4gc3RvcHdvcmRzIGFuZCBsZW4odykgPiAyXQogICAgcmV0dXJuIHNldChrZXl3b3JkcykKCgpkZWYgZXZhbHVhdGVfYW5zd2VyX3F1YWxpdHkoZXhwZWN0ZWQsIGdvdCk6CiAgICAiIiJGbGV4aWJsZSBhbnN3ZXIgZXZhbHVhdGlvbiB1c2luZyBtdWx0aXBsZSBjcml0ZXJpYS4iIiIKICAgIGV4cGVjdGVkX2xvd2VyID0gZXhwZWN0ZWQubG93ZXIoKQogICAgZ290X2xvd2VyID0gZ290Lmxvd2VyKCkKICAgIAogICAgIyAxLiBCaWRpcmVjdGlvbmFsIGNvbnRhaW5tZW50CiAgICBpZiBleHBlY3RlZF9sb3dlciBpbiBnb3RfbG93ZXIgb3IgZ290X2xvd2VyIGluIGV4cGVjdGVkX2xvd2VyOgogICAgICAgIHJldHVybiBUcnVlCiAgICAKICAgICMgMi4gUGhyYXNlIG92ZXJsYXAKICAgIGdvdF93b3JkcyA9IGdvdF9sb3dlci5zcGxpdCgpCiAgICBpZiBsZW4oZ290X3dvcmRzKSA+PSAzOgogICAgICAgIGZvciBpIGluIHJhbmdlKGxlbihnb3Rfd29yZHMpIC0gMik6CiAgICAgICAgICAgIHBocmFzZSA9ICcgJy5qb2luKGdvdF93b3Jkc1tpOmkrM10pCiAgICAgICAgICAgIGlmIHBocmFzZSBpbiBleHBlY3RlZF9sb3dlcjoKICAgICAgICAgICAgICAgIHJldHVybiBUcnVlCiAgICAKICAgICMgMy4gS2V5IGVsZW1lbnRzIG1hdGNoaW5nCiAgICBleHBlY3RlZF9rZXlzID0gZXh0cmFjdF9rZXlfZWxlbWVudHMoZXhwZWN0ZWQpCiAgICBnb3Rfa2V5cyA9IGV4dHJhY3Rfa2V5X2VsZW1lbnRzKGdvdCkKICAgIGlmIGV4cGVjdGVkX2tleXMgYW5kIGV4cGVjdGVkX2tleXMgJiBnb3Rfa2V5czoKICAgICAgICByZXR1cm4gVHJ1ZQogICAgCiAgICAjIDQuIEtleXdvcmQgb3ZlcmxhcCA+PSA0MCUKICAgIGV4cGVjdGVkX2tleXdvcmRzID0gZXh0cmFjdF9rZXl3b3JkcyhleHBlY3RlZCkKICAgIGdvdF9rZXl3b3JkcyA9IGV4dHJhY3Rfa2V5d29yZHMoZ290KQogICAgaWYgZXhwZWN0ZWRfa2V5d29yZHM6CiAgICAgICAgb3ZlcmxhcCA9IGxlbihleHBlY3RlZF9rZXl3b3JkcyAmIGdvdF9rZXl3b3JkcykKICAgICAgICBpZiBvdmVybGFwIC8gbGVuKGV4cGVjdGVkX2tleXdvcmRzKSA+PSAwLjQ6CiAgICAgICAgICAgIHJldHVybiBUcnVlCiAgICAKICAgIHJldHVybiBGYWxzZQoKCmRlZiBldmFsdWF0ZV9yZXRyaWV2ZXIocmV0cmlldmVyLCBkb2N1bWVudHMsIGRvY19lbWJlZGRpbmdzLCBxdWVzdGlvbnMsIHZlcmJvc2U9VHJ1ZSwgc2hvd19wcm9ncmVzcz1GYWxzZSk6CiAgICAiIiJFdmFsdWF0ZSBhIHJldHJpZXZlciBvbiBhIHNldCBvZiBxdWVzdGlvbnMuIiIiCiAgICBjb3JyZWN0ID0gMAogICAgdG90YWwgPSBsZW4ocXVlc3Rpb25zKQogICAgCiAgICByZXN1bHRzID0gW10KICAgIHByb2dyZXNzX2NoYXJzID0gW10KICAgIAogICAgZm9yIGksIChxdWVzdGlvbiwgZXhwZWN0ZWQpIGluIGVudW1lcmF0ZShxdWVzdGlvbnMpOgogICAgICAgIHJlc3VsdCA9IHJldHJpZXZlci5hbnN3ZXIocXVlc3Rpb24sIGRvY3VtZW50cywgZG9jX2VtYmVkZGluZ3MsIHRvcGs9MykKICAgICAgICBpc19jb3JyZWN0ID0gZXZhbHVhdGVfYW5zd2VyX3F1YWxpdHkoZXhwZWN0ZWQsIHJlc3VsdFsiYW5zd2VyIl0pCiAgICAgICAgaWYgaXNfY29ycmVjdDoKICAgICAgICAgICAgY29ycmVjdCArPSAxCiAgICAgICAgCiAgICAgICAgcmVzdWx0cy5hcHBlbmQoewogICAgICAgICAgICAicXVlc3Rpb24iOiBxdWVzdGlvbiwKICAgICAgICAgICAgImV4cGVjdGVkIjogZXhwZWN0ZWQsCiAgICAgICAgICAgICJhbnN3ZXIiOiByZXN1bHRbImFuc3dlciJdLAogICAgICAgICAgICAiY29ycmVjdCI6IGlzX2NvcnJlY3QsCiAgICAgICAgICAgICJtb2RlIjogcmVzdWx0WyJtb2RlIl0KICAgICAgICB9KQogICAgICAgIAogICAgICAgICMgU2hvdyBwcm9ncmVzcyBpbmRpY2F0b3IKICAgICAgICBpZiBzaG93X3Byb2dyZXNzOgogICAgICAgICAgICBzdGF0dXNfY2hhciA9ICIuIiBpZiBpc19jb3JyZWN0IGVsc2UgIngiCiAgICAgICAgICAgIHByb2dyZXNzX2NoYXJzLmFwcGVuZChzdGF0dXNfY2hhcikKICAgICAgICAgICAgIyBQcmludCBwcm9ncmVzcyBldmVyeSA1IHF1ZXN0aW9ucyBvciBhdCB0aGUgZW5kCiAgICAgICAgICAgIGlmIChpICsgMSkgJSAxMCA9PSAwIG9yIGkgPT0gdG90YWwgLSAxOgogICAgICAgICAgICAgICAgcHJpbnQoZiIgICAgICBbe2krMX0ve3RvdGFsfV0geycnLmpvaW4ocHJvZ3Jlc3NfY2hhcnNbLTEwOl0pfSIsIGVuZD0iXHIiKQogICAgICAgIAogICAgICAgIGlmIHZlcmJvc2U6CiAgICAgICAgICAgIHN0YXR1cyA9ICJvayIgaWYgaXNfY29ycmVjdCBlbHNlICJYIgogICAgICAgICAgICBwcmludChmIiAgICAgIFt7aSsxfS97dG90YWx9XSB7c3RhdHVzfSBROiB7cXVlc3Rpb25bOjUwXX0iKQogICAgICAgICAgICBwcmludChmIiAgICAgICAgICAgRXhwOiB7ZXhwZWN0ZWRbOjQwXX0iKQogICAgICAgICAgICBwcmludChmIiAgICAgICAgICAgR290OiB7cmVzdWx0WydhbnN3ZXInXVs6NDBdfSIpCiAgICAKICAgIGlmIHNob3dfcHJvZ3Jlc3M6CiAgICAgICAgcHJpbnQoKSAgIyBOZXcgbGluZSBhZnRlciBwcm9ncmVzcwogICAgCiAgICBhY2N1cmFjeSA9IGNvcnJlY3QgLyB0b3RhbCAqIDEwMCBpZiB0b3RhbCA+IDAgZWxzZSAwCiAgICByZXR1cm4gYWNjdXJhY3ksIHJlc3VsdHMKCgpkZWYgcnVuX3NhbXBsZV9lc3NheV9tb2RlKGFyZ3MpOgogICAgIiIiUnVuIGNvbXBhcmlzb24gb24gc2FtcGxlIGVzc2F5cyAob3JpZ2luYWwgbW9kZSkuIiIiCiAgICBFU1NBWVMsIFRFU1RfUVVFU1RJT05TID0gbG9hZF9zYW1wbGVfZXNzYXlzKCkKICAgIAogICAgcHJpbnQoIj0iICogNjApCiAgICBwcmludCgiQkFTRUxJTkUgQ09NUEFSSVNPTjogUHVyZVJBRyB2cyBUaXRhbk9ubHkgdnMgSHlicmlkUkFHIikKICAgIHByaW50KGYiTW9kZTogU2FtcGxlIEVzc2F5ICh7YXJncy5lc3NheX0pIikKICAgIHByaW50KCI9IiAqIDYwKQogICAgCiAgICAjIEluaXRpYWxpemUgY29tcG9uZW50cwogICAgcHJpbnQoIlxu8J+TpiBMb2FkaW5nIGNvbXBvbmVudHMuLi4iKQogICAgCiAgICAjIEltcG9ydCBERVZJQ0UgZnJvbSBlbWJlZGRlcnMgb3IgdGl0YW5fdXRpbHMKICAgIGZyb20gY29tbW9uLmVtYmVkZGVycyBpbXBvcnQgREVWSUNFCiAgICAKICAgIGVtYmVkZGVyID0gU2VudGVuY2VUcmFuc2Zvcm1lckVtYmVkZGVyKHRhcmdldF9kaW09MjU2LCBkZXZpY2U9REVWSUNFKQogICAgbGxtID0gRmxhblQ1R2VuZXJhdG9yKCJnb29nbGUvZmxhbi10NS1sYXJnZSIsIGRldmljZT1ERVZJQ0UpCiAgICB0aXRhbl9yYWcgPSBjcmVhdGVfdGl0YW5fcmFnKGRpbT0yNTYsIGRldmljZT1ERVZJQ0UpCiAgICAKICAgICMgTG9hZCBlc3NheQogICAgdGV4dCA9IEVTU0FZU1thcmdzLmVzc2F5XQogICAgcXVlc3Rpb25zID0gVEVTVF9RVUVTVElPTlNbYXJncy5lc3NheV0KICAgIAogICAgIyBQYXJzZSBpbnRvIHNlbWFudGljIGNodW5rcwogICAgY2h1bmtzID0gc3BsaXRfaW50b19jaHVua3ModGV4dCwgc2VudGVuY2VzX3Blcl9jaHVuaz0zLCBvdmVybGFwX3NlbnRlbmNlcz0xKQogICAgcHJpbnQoZiIgICBTcGxpdCBpbnRvIHtsZW4oY2h1bmtzKX0gc2VtYW50aWMgY2h1bmtzIikKICAgIAogICAgIyBEaWdlc3QgZG9jdW1lbnQKICAgIHByaW50KGYiXG7wn6egIERpZ2VzdGluZyBkb2N1bWVudCAoe2FyZ3MuZXBvY2hzfSBlcG9jaHMpLi4uIikKICAgIGNodW5rX2VtYmVkZGluZ3MgPSBlbWJlZGRlci5lbWJlZF9iYXRjaChjaHVua3MpCiAgICAKICAgIGZvciBlcG9jaCBpbiByYW5nZShhcmdzLmVwb2Nocyk6CiAgICAgICAgYWxsX2VtYiA9IGVtYmVkZGVyLmVtYmVkX2JhdGNoKGNodW5rcykKICAgICAgICB0aXRhbl9yYWcuZGlnZXN0X2tub3dsZWRnZShhbGxfZW1iLnVuc3F1ZWV6ZSgwKSkKICAgICAgICBpZiAoZXBvY2ggKyAxKSAlIDEwID09IDA6CiAgICAgICAgICAgIHByaW50KGYiICAgRXBvY2gge2Vwb2NoICsgMX0ve2FyZ3MuZXBvY2hzfSIpCiAgICAKICAgICMgQ3JlYXRlIHJldHJpZXZlcnMgYW5kIGV2YWx1YXRlCiAgICByZXRyaWV2ZXJzID0gY3JlYXRlX3JldHJpZXZlcnMoZW1iZWRkZXIsIGxsbSwgdGl0YW5fcmFnKQogICAgCiAgICBwcmludCgiXG4iICsgIj0iICogNjApCiAgICByZXN1bHRzX3N1bW1hcnkgPSB7fQogICAgCiAgICBmb3IgbmFtZSwgcmV0cmlldmVyIGluIHJldHJpZXZlcnMuaXRlbXMoKToKICAgICAgICBwcmludChmIlxu8J+TiiBFdmFsdWF0aW5nOiB7bmFtZS51cHBlcigpfSIpCiAgICAgICAgcHJpbnQoIi0iICogNDApCiAgICAgICAgYWNjdXJhY3ksIHJlc3VsdHMgPSBldmFsdWF0ZV9yZXRyaWV2ZXIocmV0cmlldmVyLCBjaHVua3MsIGNodW5rX2VtYmVkZGluZ3MsIHF1ZXN0aW9ucykKICAgICAgICByZXN1bHRzX3N1bW1hcnlbbmFtZV0gPSBhY2N1cmFjeQogICAgICAgIHByaW50KGYiXG4gICBBY2N1cmFjeToge2FjY3VyYWN5Oi4xZn0lIikKICAgIAogICAgIyBTdW1tYXJ5CiAgICBwcmludCgiXG4iICsgIj0iICogNjApCiAgICBwcmludCgiU1VNTUFSWSIpCiAgICBwcmludCgiPSIgKiA2MCkKICAgIGZvciBuYW1lLCBhY2N1cmFjeSBpbiByZXN1bHRzX3N1bW1hcnkuaXRlbXMoKToKICAgICAgICBiYXIgPSAi4paIIiAqIGludChhY2N1cmFjeSAvIDEwKSArICLilpEiICogKDEwIC0gaW50KGFjY3VyYWN5IC8gMTApKQogICAgICAgIHByaW50KGYiICB7bmFtZToxMnN9OiB7YmFyfSB7YWNjdXJhY3k6LjFmfSUiKQoKCmRlZiBydW5fc3F1YWRfbW9kZShhcmdzKToKICAgICIiIlJ1biBjb21wYXJpc29uIG9uIFNRdUFEIGRhdGFzZXQgKG5ldyBtb2RlKS4iIiIKICAgIENPTlRFWFRTLCBURVNUX1FVRVNUSU9OUyA9IGxvYWRfc3F1YWRfZGF0YSgpCiAgICAKICAgIHByaW50KCI9IiAqIDYwKQogICAgcHJpbnQoIkJBU0VMSU5FIENPTVBBUklTT046IFB1cmVSQUcgdnMgVGl0YW5Pbmx5IHZzIEh5YnJpZFJBRyIpCiAgICBwcmludChmIk1vZGU6IFNRdUFEIERhdGFzZXQgKHthcmdzLnRpdGxlc30gdGl0bGVzKSIpCiAgICBwcmludCgiPSIgKiA2MCkKICAgIAogICAgIyBJbml0aWFsaXplIGNvbXBvbmVudHMKICAgIHByaW50KCJcbvCfk6YgTG9hZGluZyBjb21wb25lbnRzLi4uIikKICAgIAogICAgZnJvbSBjb21tb24uZW1iZWRkZXJzIGltcG9ydCBERVZJQ0UKICAgIAogICAgZW1iZWRkZXIgPSBTZW50ZW5jZVRyYW5zZm9ybWVyRW1iZWRkZXIodGFyZ2V0X2RpbT0yNTYsIGRldmljZT1ERVZJQ0UpCiAgICBsbG0gPSBGbGFuVDVHZW5lcmF0b3IoImdvb2dsZS9mbGFuLXQ1LWxhcmdlIiwgZGV2aWNlPURFVklDRSkKICAgIAogICAgIyBTZWxlY3QgdGl0bGVzIHRvIGV2YWx1YXRlCiAgICBhbGxfdGl0bGVzID0gbGlzdChDT05URVhUUy5rZXlzKCkpCiAgICBpZiBhcmdzLnRpdGxlcyA+PSBsZW4oYWxsX3RpdGxlcyk6CiAgICAgICAgc2VsZWN0ZWRfdGl0bGVzID0gYWxsX3RpdGxlcwogICAgZWxzZToKICAgICAgICAjIFJhbmRvbSBzYW1wbGUKICAgICAgICByYW5kb20uc2VlZCg0MikgICMgRm9yIHJlcHJvZHVjaWJpbGl0eQogICAgICAgIHNlbGVjdGVkX3RpdGxlcyA9IHJhbmRvbS5zYW1wbGUoYWxsX3RpdGxlcywgYXJncy50aXRsZXMpCiAgICAKICAgIHByaW50KGYiICAgVG90YWwgdGl0bGVzIGF2YWlsYWJsZToge2xlbihhbGxfdGl0bGVzKX0iKQogICAgcHJpbnQoZiIgICBTZWxlY3RlZCBmb3IgZXZhbHVhdGlvbjoge2xlbihzZWxlY3RlZF90aXRsZXMpfSIpCiAgICAKICAgICMgQWdncmVnYXRlIHJlc3VsdHMKICAgIGFsbF9yZXN1bHRzID0gewogICAgICAgICJwdXJlX3JhZyI6IHsiY29ycmVjdCI6IDAsICJ0b3RhbCI6IDB9LAogICAgICAgICJ0aXRhbl9vbmx5IjogeyJjb3JyZWN0IjogMCwgInRvdGFsIjogMH0sCiAgICAgICAgImh5YnJpZCI6IHsiY29ycmVjdCI6IDAsICJ0b3RhbCI6IDB9CiAgICB9CiAgICAKICAgICMgUHJvY2VzcyBlYWNoIHRpdGxlCiAgICBmb3IgaSwgdGl0bGUgaW4gZW51bWVyYXRlKHNlbGVjdGVkX3RpdGxlcyk6CiAgICAgICAgcHJpbnQoZiJcbnsnPScqNjB9IikKICAgICAgICBwcmludChmIlt7aSsxfS97bGVuKHNlbGVjdGVkX3RpdGxlcyl9XSBUaXRsZToge3RpdGxlWzo1MF19Li4uIikKICAgICAgICBwcmludCgiPSIgKiA2MCkKICAgICAgICAKICAgICAgICBjb250ZXh0ID0gQ09OVEVYVFNbdGl0bGVdCiAgICAgICAgcXVlc3Rpb25zID0gVEVTVF9RVUVTVElPTlMuZ2V0KHRpdGxlLCBbXSkKICAgICAgICAKICAgICAgICBpZiBub3QgcXVlc3Rpb25zOgogICAgICAgICAgICBwcmludCgiICAg4pqg77iPIE5vIHF1ZXN0aW9ucyBmb3IgdGhpcyB0aXRsZSwgc2tpcHBpbmcuLi4iKQogICAgICAgICAgICBjb250aW51ZQogICAgICAgIAogICAgICAgICMgTGltaXQgcXVlc3Rpb25zIHBlciB0aXRsZSB0byBhdm9pZCBsb25nIHJ1bnMKICAgICAgICBpZiBsZW4ocXVlc3Rpb25zKSA+IGFyZ3MubWF4X3F1ZXN0aW9uczoKICAgICAgICAgICAgcXVlc3Rpb25zID0gcXVlc3Rpb25zWzphcmdzLm1heF9xdWVzdGlvbnNdCiAgICAgICAgCiAgICAgICAgcHJpbnQoZiIgICBDb250ZXh0IGxlbmd0aDoge2xlbihjb250ZXh0KX0gY2hhcnMiKQogICAgICAgIHByaW50KGYiICAgUXVlc3Rpb25zOiB7bGVuKHF1ZXN0aW9ucyl9IikKICAgICAgICAKICAgICAgICAjIENyZWF0ZSBmcmVzaCBUaXRhbiBtb2RlbCBmb3IgZWFjaCB0aXRsZQogICAgICAgIHRpdGFuX3JhZyA9IGNyZWF0ZV90aXRhbl9yYWcoZGltPTI1NikKICAgICAgICAKICAgICAgICAjIFBhcnNlIGNvbnRleHQgaW50byBjaHVua3MgKGZpeGVkIHNpemUgZm9yIHN0YWJpbGl0eSkKICAgICAgICBjaHVua3MgPSBzcGxpdF9pbnRvX2NodW5rcyhjb250ZXh0LCBzZW50ZW5jZXNfcGVyX2NodW5rPTMsIG92ZXJsYXBfc2VudGVuY2VzPTEpCiAgICAgICAgaWYgbm90IGNodW5rczoKICAgICAgICAgICAgIyBGYWxsYmFjazogc3BsaXQgYnkgcGFyYWdyYXBocwogICAgICAgICAgICBjaHVua3MgPSBbcC5zdHJpcCgpIGZvciBwIGluIGNvbnRleHQuc3BsaXQoJ1xuXG4nKSBpZiBwLnN0cmlwKCldCiAgICAgICAgaWYgbm90IGNodW5rczoKICAgICAgICAgICAgY2h1bmtzID0gW2NvbnRleHRdCiAgICAgICAgCiAgICAgICAgcHJpbnQoZiIgICBDaHVua3M6IHtsZW4oY2h1bmtzKX0iKQogICAgICAgIAogICAgICAgICMgRW1iZWQgY2h1bmtzCiAgICAgICAgY2h1bmtfZW1iZWRkaW5ncyA9IGVtYmVkZGVyLmVtYmVkX2JhdGNoKGNodW5rcykKICAgICAgICAKICAgICAgICAjIERpZ2VzdCBkb2N1bWVudCAoZml4ZWQgZXBvY2hzIGZvciBzdGFiaWxpdHkpCiAgICAgICAgZGlnZXN0X2Vwb2NocyA9IG1pbihhcmdzLmVwb2NocywgNTAwKQogICAgICAgIHByaW50KGYiICAgRGlnZXN0aW5nICh7ZGlnZXN0X2Vwb2Noc30gZXBvY2hzKTogIiwgZW5kPSIiLCBmbHVzaD1UcnVlKQogICAgICAgIGZvciBlcG9jaCBpbiByYW5nZShkaWdlc3RfZXBvY2hzKToKICAgICAgICAgICAgYWxsX2VtYiA9IGVtYmVkZGVyLmVtYmVkX2JhdGNoKGNodW5rcykKICAgICAgICAgICAgdGl0YW5fcmFnLmRpZ2VzdF9rbm93bGVkZ2UoYWxsX2VtYi51bnNxdWVlemUoMCkpCiAgICAgICAgICAgIGlmIChlcG9jaCArIDEpICUgMTAgPT0gMDoKICAgICAgICAgICAgICAgIHByaW50KGYie2Vwb2NoKzF9IiwgZW5kPSIgIiwgZmx1c2g9VHJ1ZSkKICAgICAgICBwcmludCgiZG9uZSIpCiAgICAgICAgCiAgICAgICAgIyBDcmVhdGUgcmV0cmlldmVycwogICAgICAgIHJldHJpZXZlcnMgPSBjcmVhdGVfcmV0cmlldmVycyhlbWJlZGRlciwgbGxtLCB0aXRhbl9yYWcpCiAgICAgICAgCiAgICAgICAgIyBFdmFsdWF0ZSBlYWNoIHJldHJpZXZlciB3aXRoIHByb2dyZXNzCiAgICAgICAgZm9yIG5hbWUsIHJldHJpZXZlciBpbiByZXRyaWV2ZXJzLml0ZW1zKCk6CiAgICAgICAgICAgIHByaW50KGYiICAgW3tuYW1lfV0gRXZhbHVhdGluZy4uLiIsIGVuZD0iIiwgZmx1c2g9VHJ1ZSkKICAgICAgICAgICAgYWNjdXJhY3ksIHJlc3VsdHMgPSBldmFsdWF0ZV9yZXRyaWV2ZXIoCiAgICAgICAgICAgICAgICByZXRyaWV2ZXIsIGNodW5rcywgY2h1bmtfZW1iZWRkaW5ncywgcXVlc3Rpb25zLCAKICAgICAgICAgICAgICAgIHZlcmJvc2U9YXJncy52ZXJib3NlLCBzaG93X3Byb2dyZXNzPW5vdCBhcmdzLnZlcmJvc2UKICAgICAgICAgICAgKQogICAgICAgICAgICAKICAgICAgICAgICAgY29ycmVjdF9jb3VudCA9IHN1bSgxIGZvciByIGluIHJlc3VsdHMgaWYgclsiY29ycmVjdCJdKQogICAgICAgICAgICBhbGxfcmVzdWx0c1tuYW1lXVsiY29ycmVjdCJdICs9IGNvcnJlY3RfY291bnQKICAgICAgICAgICAgYWxsX3Jlc3VsdHNbbmFtZV1bInRvdGFsIl0gKz0gbGVuKHF1ZXN0aW9ucykKICAgICAgICAgICAgCiAgICAgICAgICAgIHByaW50KGYiIHtjb3JyZWN0X2NvdW50fS97bGVuKHF1ZXN0aW9ucyl9ICh7YWNjdXJhY3k6LjFmfSUpIikKICAgIAogICAgIyBGaW5hbCBTdW1tYXJ5CiAgICBwcmludCgiXG4iICsgIj0iICogNjApCiAgICBwcmludCgiRklOQUwgU1VNTUFSWSIpCiAgICBwcmludCgiPSIgKiA2MCkKICAgIHByaW50KGYiRXZhbHVhdGVkIG9uIHtsZW4oc2VsZWN0ZWRfdGl0bGVzKX0gdGl0bGVzIikKICAgIHByaW50KCkKICAgIAogICAgZm9yIG5hbWUsIHN0YXRzIGluIGFsbF9yZXN1bHRzLml0ZW1zKCk6CiAgICAgICAgaWYgc3RhdHNbInRvdGFsIl0gPiAwOgogICAgICAgICAgICBhY2N1cmFjeSA9IHN0YXRzWyJjb3JyZWN0Il0gLyBzdGF0c1sidG90YWwiXSAqIDEwMAogICAgICAgICAgICBiYXIgPSAi4paIIiAqIGludChhY2N1cmFjeSAvIDEwKSArICLilpEiICogKDEwIC0gaW50KGFjY3VyYWN5IC8gMTApKQogICAgICAgICAgICBwcmludChmIiAge25hbWU6MTJzfToge2Jhcn0ge2FjY3VyYWN5Oi4xZn0lICh7c3RhdHNbJ2NvcnJlY3QnXX0ve3N0YXRzWyd0b3RhbCddfSkiKQoKCmRlZiBtYWluKCk6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj0iQ29tcGFyZSBiYXNlbGluZSByZXRyaWV2YWwgc3RyYXRlZ2llcyIpCiAgICAKICAgICMgTW9kZSBzZWxlY3Rpb24KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tc3F1YWQiLCBhY3Rpb249InN0b3JlX3RydWUiLCAKICAgICAgICAgICAgICAgICAgICAgICBoZWxwPSJVc2UgU1F1QUQgZGF0YXNldCBpbnN0ZWFkIG9mIHNhbXBsZSBlc3NheXMiKQogICAgCiAgICAjIFNhbXBsZSBlc3NheSBtb2RlIG9wdGlvbnMKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZXNzYXkiLCB0eXBlPXN0ciwgZGVmYXVsdD0iY2xpbWF0ZSIsCiAgICAgICAgICAgICAgICAgICAgICAgaGVscD0iRXNzYXkgdG8gdXNlIChjbGltYXRlL2FpL3NwYWNlKSIpCiAgICAKICAgICMgU1F1QUQgbW9kZSBvcHRpb25zCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXRpdGxlcyIsIHR5cGU9aW50LCBkZWZhdWx0PTEwLAogICAgICAgICAgICAgICAgICAgICAgIGhlbHA9Ik51bWJlciBvZiB0aXRsZXMgdG8gZXZhbHVhdGUgKFNRdUFEIG1vZGUpIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbWF4LXF1ZXN0aW9ucyIsIHR5cGU9aW50LCBkZWZhdWx0PTUsCiAgICAgICAgICAgICAgICAgICAgICAgaGVscD0iTWF4IHF1ZXN0aW9ucyBwZXIgdGl0bGUgKFNRdUFEIG1vZGUpIikKICAgIAogICAgIyBDb21tb24gb3B0aW9ucwogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1lcG9jaHMiLCB0eXBlPWludCwgZGVmYXVsdD01MCwgaGVscD0iRGlnZXN0aW9uIGVwb2NocyIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXRvcGsiLCB0eXBlPWludCwgZGVmYXVsdD01LCBoZWxwPSJUb3AtSyBmb3IgcmV0cmlldmFsIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tdmVyYm9zZSIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIsIGhlbHA9IlNob3cgZGV0YWlsZWQgb3V0cHV0IikKICAgIAogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkKICAgIAogICAgaWYgYXJncy5zcXVhZDoKICAgICAgICBydW5fc3F1YWRfbW9kZShhcmdzKQogICAgZWxzZToKICAgICAgICBydW5fc2FtcGxlX2Vzc2F5X21vZGUoYXJncykKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg=='
}

print(f'Setting up {len(files)} core files...')
for path, content in files.items():
    write_file(path, content)

print('\n✅ Core files ready!')


## 2.5 Setup SQuAD Dataset (Optional)

This cell will automatically download and generate the SQuAD dataset.
**Skip this cell if you only want to use Sample Essays mode.**

In [ ]:
import os
import base64

# Check if SQuAD data already exists
if os.path.exists('data/processed_squad.py'):
    size_mb = os.path.getsize('data/processed_squad.py') / (1024 * 1024)
    print(f'✅ SQuAD dataset already exists ({size_mb:.1f} MB)')
else:
    print('📥 Downloading SQuAD dataset...')
    !wget -q --show-progress https://rajpurkar.github.io/SQuAD-explorer/dataset/train-v2.0.json -O data/train-v2.0.json
    
    # Write the processing script
    script_b64 = 'IiIiClNjcmlwdCB0byBjbGVhbiBhbmQgcHJvY2VzcyBTUXVBRCBkYXRhc2V0IChKU09OIG9yIENTViBmb3JtYXQpLgoKVGhpcyBzY3JpcHQ6CjEuIEZpbHRlcnMgb3V0IHVuaXF1ZSB0aXRsZXMKMi4gQ29tYmluZXMgZHVwbGljYXRlIGNvbnRleHRzIGJhc2VkIG9uIHRpdGxlCjMuIEV4dHJhY3RzIHF1ZXN0aW9uIGFuZCBhbnN3ZXIgcGFpcnMgZm9yIGVhY2ggY29udGV4dAoKT3V0cHV0IGZvcm1hdCBtYXRjaGVzIHNhbXBsZV9lc3NheXMucHkgc3RydWN0dXJlLgoKU3VwcG9ydHM6Ci0gU1F1QUQgdjIuMCBKU09OIGZvcm1hdCAodHJhaW4tdjIuMC5qc29uKQotIENTViBmb3JtYXQgKHRyYWluLmNzdikKIiIiCgppbXBvcnQgY3N2CmltcG9ydCBhc3QKaW1wb3J0IHJlCmltcG9ydCBzeXMKaW1wb3J0IGlvCmltcG9ydCBqc29uCmltcG9ydCBvcwoKIyBGaXggV2luZG93cyBjb25zb2xlIGVuY29kaW5nCnN5cy5zdGRvdXQgPSBpby5UZXh0SU9XcmFwcGVyKHN5cy5zdGRvdXQuYnVmZmVyLCBlbmNvZGluZz0ndXRmLTgnKQoKCmRlZiBsb2FkX2Nzdl9kYXRhKGZpbGVfcGF0aDogc3RyKSAtPiBsaXN0OgogICAgIiIiCiAgICBMb2FkIENTViBkYXRhc2V0LgogICAgCiAgICBBcmdzOgogICAgICAgIGZpbGVfcGF0aDogUGF0aCB0byB0aGUgQ1NWIGZpbGUKICAgICAgICAKICAgIFJldHVybnM6CiAgICAgICAgTGlzdCBvZiByb3cgZGljdGlvbmFyaWVzCiAgICAiIiIKICAgIGRhdGEgPSBbXQogICAgd2l0aCBvcGVuKGZpbGVfcGF0aCwgJ3InLCBlbmNvZGluZz0ndXRmLTgnKSBhcyBmOgogICAgICAgIHJlYWRlciA9IGNzdi5EaWN0UmVhZGVyKGYpCiAgICAgICAgZm9yIHJvdyBpbiByZWFkZXI6CiAgICAgICAgICAgIGRhdGEuYXBwZW5kKHJvdykKICAgIHByaW50KGYiTG9hZGVkIHtsZW4oZGF0YSl9IHJvd3MgZnJvbSBDU1YiKQogICAgcmV0dXJuIGRhdGEKCgpkZWYgbG9hZF9qc29uX2RhdGEoZmlsZV9wYXRoOiBzdHIpIC0+IGxpc3Q6CiAgICAiIiIKICAgIExvYWQgU1F1QUQgdjIuMCBKU09OIGRhdGFzZXQgYW5kIGNvbnZlcnQgdG8gcm93IGZvcm1hdC4KICAgIAogICAgQXJnczoKICAgICAgICBmaWxlX3BhdGg6IFBhdGggdG8gdGhlIEpTT04gZmlsZSAodHJhaW4tdjIuMC5qc29uKQogICAgICAgIAogICAgUmV0dXJuczoKICAgICAgICBMaXN0IG9mIHJvdyBkaWN0aW9uYXJpZXMgd2l0aCBrZXlzOiB0aXRsZSwgY29udGV4dCwgcXVlc3Rpb24sIGFuc3dlcnMKICAgICIiIgogICAgd2l0aCBvcGVuKGZpbGVfcGF0aCwgJ3InLCBlbmNvZGluZz0ndXRmLTgnKSBhcyBmOgogICAgICAgIHNxdWFkX2RhdGEgPSBqc29uLmxvYWQoZikKICAgIAogICAgZGF0YSA9IFtdCiAgICBmb3IgYXJ0aWNsZSBpbiBzcXVhZF9kYXRhWydkYXRhJ106CiAgICAgICAgdGl0bGUgPSBhcnRpY2xlWyd0aXRsZSddCiAgICAgICAgZm9yIHBhcmFncmFwaCBpbiBhcnRpY2xlWydwYXJhZ3JhcGhzJ106CiAgICAgICAgICAgIGNvbnRleHQgPSBwYXJhZ3JhcGhbJ2NvbnRleHQnXQogICAgICAgICAgICBmb3IgcWEgaW4gcGFyYWdyYXBoWydxYXMnXToKICAgICAgICAgICAgICAgIHF1ZXN0aW9uID0gcWFbJ3F1ZXN0aW9uJ10KICAgICAgICAgICAgICAgICMgSGFuZGxlIGJvdGggdjEgYW5kIHYyIGZvcm1hdAogICAgICAgICAgICAgICAgYW5zd2VycyA9IHFhLmdldCgnYW5zd2VycycsIFtdKQogICAgICAgICAgICAgICAgaWYgbm90IGFuc3dlcnMgYW5kICdwbGF1c2libGVfYW5zd2VycycgaW4gcWE6CiAgICAgICAgICAgICAgICAgICAgYW5zd2VycyA9IHFhWydwbGF1c2libGVfYW5zd2VycyddCiAgICAgICAgICAgICAgICAKICAgICAgICAgICAgICAgICMgQ29udmVydCB0byBmb3JtYXQgZXhwZWN0ZWQgYnkgcHJvY2Vzc19kYXRhc2V0CiAgICAgICAgICAgICAgICBhbnN3ZXJfdGV4dCA9IGFuc3dlcnNbMF1bJ3RleHQnXSBpZiBhbnN3ZXJzIGVsc2UgJycKICAgICAgICAgICAgICAgIGRhdGEuYXBwZW5kKHsKICAgICAgICAgICAgICAgICAgICAndGl0bGUnOiB0aXRsZSwKICAgICAgICAgICAgICAgICAgICAnY29udGV4dCc6IGNvbnRleHQsCiAgICAgICAgICAgICAgICAgICAgJ3F1ZXN0aW9uJzogcXVlc3Rpb24sCiAgICAgICAgICAgICAgICAgICAgJ2Fuc3dlcnMnOiBhbnN3ZXJfdGV4dCwgICMgQWxyZWFkeSBwYXJzZWQKICAgICAgICAgICAgICAgICAgICAnX3BhcnNlZCc6IFRydWUgICMgRmxhZyB0byBza2lwIHBhcnNlX2Fuc3dlcgogICAgICAgICAgICAgICAgfSkKICAgIAogICAgcHJpbnQoZiJMb2FkZWQge2xlbihkYXRhKX0gUUEgcGFpcnMgZnJvbSBKU09OIikKICAgIHJldHVybiBkYXRhCgoKZGVmIHBhcnNlX2Fuc3dlcihhbnN3ZXJzX3N0cjogc3RyKSAtPiBzdHI6CiAgICAiIiIKICAgIFBhcnNlIHRoZSBhbnN3ZXIgc3RyaW5nIGZyb20gQ1NWIGZvcm1hdC4KICAgIAogICAgRm9ybWF0OiB7J3RleHQnOiBhcnJheShbJ2Fuc3dlciB0ZXh0J10sIGR0eXBlPW9iamVjdCksICdhbnN3ZXJfc3RhcnQnOiBhcnJheShbMTIzXSwgZHR5cGU9aW50MzIpfQogICAgCiAgICBBcmdzOgogICAgICAgIGFuc3dlcnNfc3RyOiBSYXcgYW5zd2VyIHN0cmluZyBmcm9tIENTVgogICAgICAgIAogICAgUmV0dXJuczoKICAgICAgICBFeHRyYWN0ZWQgYW5zd2VyIHRleHQsIG9yIGVtcHR5IHN0cmluZyBpZiBwYXJzaW5nIGZhaWxzCiAgICAiIiIKICAgIHRyeToKICAgICAgICAjIENsZWFuIG51bXB5IGFycmF5IGZvcm1hdCB0byBQeXRob24gbGlzdCBmb3JtYXQKICAgICAgICBjbGVhbmVkID0gYW5zd2Vyc19zdHIucmVwbGFjZSgiYXJyYXkoIiwgIiIpLnJlcGxhY2UoIiwgZHR5cGU9b2JqZWN0KSIsICIiKS5yZXBsYWNlKCIsIGR0eXBlPWludDMyKSIsICIiKQogICAgICAgIGFuc3dlcnNfZGljdCA9IGFzdC5saXRlcmFsX2V2YWwoY2xlYW5lZCkKICAgICAgICAKICAgICAgICB0ZXh0X2xpc3QgPSBhbnN3ZXJzX2RpY3QuZ2V0KCd0ZXh0JywgW10pCiAgICAgICAgaWYgdGV4dF9saXN0IGFuZCBsZW4odGV4dF9saXN0KSA+IDA6CiAgICAgICAgICAgIHJldHVybiB0ZXh0X2xpc3RbMF0uc3RyaXAoKQogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICBwYXNzCiAgICByZXR1cm4gIiIKCgpkZWYgcHJvY2Vzc19kYXRhc2V0KGRhdGE6IGxpc3QpIC0+IHR1cGxlOgogICAgIiIiCiAgICBQcm9jZXNzIHRoZSBkYXRhc2V0IHRvIGV4dHJhY3QgdW5pcXVlIHRpdGxlcywgY29tYmluZWQgY29udGV4dHMsIGFuZCBRQSBwYWlycy4KICAgIAogICAgQXJnczoKICAgICAgICBkYXRhOiBMaXN0IG9mIHJvdyBkaWN0aW9uYXJpZXMgZnJvbSBDU1Ygb3IgSlNPTgogICAgICAgIAogICAgUmV0dXJuczoKICAgICAgICBUdXBsZSBvZiAoY29udGV4dHNfZGljdCwgcXVlc3Rpb25zX2RpY3QpCiAgICAgICAgLSBjb250ZXh0c19kaWN0OiB7dGl0bGU6IGNvbWJpbmVkX2NvbnRleHR9CiAgICAgICAgLSBxdWVzdGlvbnNfZGljdDoge3RpdGxlOiBbKHF1ZXN0aW9uLCBhbnN3ZXIpLCAuLi5dfQogICAgIiIiCiAgICAjIFN0b3JlIHVuaXF1ZSBjb250ZXh0cyBwZXIgdGl0bGUgKHByZXNlcnZlIG9yZGVyLCBhdm9pZCBkdXBsaWNhdGVzKQogICAgdGl0bGVfY29udGV4dHMgPSB7fSAgIyB7dGl0bGU6IFtsaXN0IG9mIHVuaXF1ZSBjb250ZXh0IHBhcmFncmFwaHNdfQogICAgdGl0bGVfcXVlc3Rpb25zID0ge30gICMge3RpdGxlOiBbKHF1ZXN0aW9uLCBhbnN3ZXIpLCAuLi5dfQogICAgCiAgICBmb3Igcm93IGluIGRhdGE6CiAgICAgICAgdGl0bGUgPSByb3dbJ3RpdGxlJ10uc3RyaXAoKQogICAgICAgIGNvbnRleHQgPSByb3dbJ2NvbnRleHQnXS5zdHJpcCgpCiAgICAgICAgcXVlc3Rpb24gPSByb3dbJ3F1ZXN0aW9uJ10uc3RyaXAoKQogICAgICAgIAogICAgICAgICMgQ2hlY2sgaWYgYW5zd2VyIGlzIHByZS1wYXJzZWQgKGZyb20gSlNPTikgb3IgbmVlZHMgcGFyc2luZyAoZnJvbSBDU1YpCiAgICAgICAgaWYgcm93LmdldCgnX3BhcnNlZCcpOgogICAgICAgICAgICBhbnN3ZXIgPSByb3dbJ2Fuc3dlcnMnXS5zdHJpcCgpIGlmIHJvd1snYW5zd2VycyddIGVsc2UgJycKICAgICAgICBlbHNlOgogICAgICAgICAgICBhbnN3ZXIgPSBwYXJzZV9hbnN3ZXIocm93WydhbnN3ZXJzJ10pCiAgICAgICAgCiAgICAgICAgIyBJbml0aWFsaXplIHRpdGxlIGVudHJ5IGlmIG5vdCBleGlzdHMKICAgICAgICBpZiB0aXRsZSBub3QgaW4gdGl0bGVfY29udGV4dHM6CiAgICAgICAgICAgIHRpdGxlX2NvbnRleHRzW3RpdGxlXSA9IFtdCiAgICAgICAgICAgIHRpdGxlX3F1ZXN0aW9uc1t0aXRsZV0gPSBbXQogICAgICAgIAogICAgICAgICMgQWRkIGNvbnRleHQgaWYgbm90IGFscmVhZHkgcHJlc2VudCAoYXZvaWQgZHVwbGljYXRlcykKICAgICAgICBpZiBjb250ZXh0IG5vdCBpbiB0aXRsZV9jb250ZXh0c1t0aXRsZV06CiAgICAgICAgICAgIHRpdGxlX2NvbnRleHRzW3RpdGxlXS5hcHBlbmQoY29udGV4dCkKICAgICAgICAKICAgICAgICAjIEFkZCBxdWVzdGlvbi1hbnN3ZXIgcGFpciAob25seSBpZiBhbnN3ZXIgZXhpc3RzKQogICAgICAgIGlmIGFuc3dlcjoKICAgICAgICAgICAgdGl0bGVfcXVlc3Rpb25zW3RpdGxlXS5hcHBlbmQoKHF1ZXN0aW9uLCBhbnN3ZXIpKQogICAgCiAgICAjIENvbWJpbmUgY29udGV4dHMgZm9yIGVhY2ggdGl0bGUKICAgIGNvbnRleHRzX2RpY3QgPSB7fQogICAgZm9yIHRpdGxlLCBwYXJhZ3JhcGhzIGluIHRpdGxlX2NvbnRleHRzLml0ZW1zKCk6CiAgICAgICAgY29udGV4dHNfZGljdFt0aXRsZV0gPSAiXG5cbiIuam9pbihwYXJhZ3JhcGhzKQogICAgCiAgICBwcmludChmIlByb2Nlc3NlZCB7bGVuKGNvbnRleHRzX2RpY3QpfSB1bmlxdWUgdGl0bGVzIikKICAgIHByaW50KGYiVG90YWwgUUEgcGFpcnM6IHtzdW0obGVuKHFhKSBmb3IgcWEgaW4gdGl0bGVfcXVlc3Rpb25zLnZhbHVlcygpKX0iKQogICAgCiAgICByZXR1cm4gY29udGV4dHNfZGljdCwgdGl0bGVfcXVlc3Rpb25zCgoKZGVmIGdlbmVyYXRlX291dHB1dF9maWxlKGNvbnRleHRzOiBkaWN0LCBxdWVzdGlvbnM6IGRpY3QsIG91dHB1dF9wYXRoOiBzdHIpOgogICAgIiIiCiAgICBHZW5lcmF0ZSBvdXRwdXQgUHl0aG9uIGZpbGUgaW4gc2FtcGxlX2Vzc2F5cy5weSBmb3JtYXQuCiAgICAKICAgIEFyZ3M6CiAgICAgICAgY29udGV4dHM6IERpY3Rpb25hcnkgb2Yge3RpdGxlOiBjb21iaW5lZF9jb250ZXh0fQogICAgICAgIHF1ZXN0aW9uczogRGljdGlvbmFyeSBvZiB7dGl0bGU6IFsocXVlc3Rpb24sIGFuc3dlciksIC4uLl19CiAgICAgICAgb3V0cHV0X3BhdGg6IFBhdGggZm9yIG91dHB1dCBmaWxlCiAgICAiIiIKICAgIHdpdGggb3BlbihvdXRwdXRfcGF0aCwgJ3cnLCBlbmNvZGluZz0ndXRmLTgnKSBhcyBmOgogICAgICAgICMgSGVhZGVyCiAgICAgICAgZi53cml0ZSgnIiIiXG4nKQogICAgICAgIGYud3JpdGUoJ1Byb2Nlc3NlZCBTUXVBRCBEYXRhc2V0IGZvciBUaXRhblJBRyBFeHBlcmltZW50c1xuJykKICAgICAgICBmLndyaXRlKCdcbicpCiAgICAgICAgZi53cml0ZSgnVGhpcyBmaWxlIGNvbnRhaW5zIHByb2Nlc3NlZCBjb250ZXh0cyBhbmQgcXVlc3Rpb24tYW5zd2VyIHBhaXJzXG4nKQogICAgICAgIGYud3JpdGUoJ2V4dHJhY3RlZCBmcm9tIHRoZSB0cmFpbi5jc3YgZGF0YXNldC5cbicpCiAgICAgICAgZi53cml0ZSgnIiIiXG5cbicpCiAgICAgICAgCiAgICAgICAgIyBHZW5lcmF0ZSBjb250ZXh0IHZhcmlhYmxlcwogICAgICAgIGYud3JpdGUoJyMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09XG4nKQogICAgICAgIGYud3JpdGUoJyMgQ09OVEVYVFMgLSBDb21iaW5lZCBwYXJhZ3JhcGhzIGJ5IHRpdGxlXG4nKQogICAgICAgIGYud3JpdGUoJyMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09XG5cbicpCiAgICAgICAgCiAgICAgICAgZm9yIHRpdGxlLCBjb250ZXh0IGluIGNvbnRleHRzLml0ZW1zKCk6CiAgICAgICAgICAgICMgQ3JlYXRlIHZhbGlkIFB5dGhvbiB2YXJpYWJsZSBuYW1lCiAgICAgICAgICAgIHZhcl9uYW1lID0gY3JlYXRlX3ZhcmlhYmxlX25hbWUodGl0bGUpCiAgICAgICAgICAgICMgRXNjYXBlIHRyaXBsZSBxdW90ZXMgaW4gY29udGVudAogICAgICAgICAgICBlc2NhcGVkX2NvbnRleHQgPSBjb250ZXh0LnJlcGxhY2UoJyIiIicsICdcXCJcXCJcXCInKQogICAgICAgICAgICBmLndyaXRlKGYne3Zhcl9uYW1lfSA9ICIiIlxue2VzY2FwZWRfY29udGV4dH1cbiIiIlxuXG4nKQogICAgICAgIAogICAgICAgICMgR2VuZXJhdGUgcXVlc3Rpb25zIGRpY3Rpb25hcnkKICAgICAgICBmLndyaXRlKCcjID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PVxuJykKICAgICAgICBmLndyaXRlKCcjIFRFU1RfUVVFU1RJT05TIC0gUXVlc3Rpb24tQW5zd2VyIHBhaXJzIGJ5IHRpdGxlXG4nKQogICAgICAgIGYud3JpdGUoJyMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09XG5cbicpCiAgICAgICAgZi53cml0ZSgnVEVTVF9RVUVTVElPTlMgPSB7XG4nKQogICAgICAgIAogICAgICAgIGZvciB0aXRsZSwgcWFfcGFpcnMgaW4gcXVlc3Rpb25zLml0ZW1zKCk6CiAgICAgICAgICAgIGlmIG5vdCBxYV9wYWlyczoKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgICMgRXNjYXBlIHF1b3RlcyBpbiB0aXRsZQogICAgICAgICAgICBlc2NhcGVkX3RpdGxlID0gdGl0bGUucmVwbGFjZSgnIicsICdcXCInKQogICAgICAgICAgICBmLndyaXRlKGYnICAgICJ7ZXNjYXBlZF90aXRsZX0iOiBbXG4nKQogICAgICAgICAgICAKICAgICAgICAgICAgZm9yIHF1ZXN0aW9uLCBhbnN3ZXIgaW4gcWFfcGFpcnM6CiAgICAgICAgICAgICAgICAjIEVzY2FwZSBxdW90ZXMgaW4gcXVlc3Rpb24gYW5kIGFuc3dlcgogICAgICAgICAgICAgICAgZXNjYXBlZF9xID0gcXVlc3Rpb24ucmVwbGFjZSgnIicsICdcXCInKS5yZXBsYWNlKCdcbicsICcgJykKICAgICAgICAgICAgICAgIGVzY2FwZWRfYSA9IGFuc3dlci5yZXBsYWNlKCciJywgJ1xcIicpLnJlcGxhY2UoJ1xuJywgJyAnKQogICAgICAgICAgICAgICAgZi53cml0ZShmJyAgICAgICAgKCJ7ZXNjYXBlZF9xfSIsICJ7ZXNjYXBlZF9hfSIpLFxuJykKICAgICAgICAgICAgCiAgICAgICAgICAgIGYud3JpdGUoJyAgICBdLFxuJykKICAgICAgICAKICAgICAgICBmLndyaXRlKCd9XG5cbicpCiAgICAgICAgCiAgICAgICAgIyBIZWxwZXIgZnVuY3Rpb25zCiAgICAgICAgZi53cml0ZSgnIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT1cbicpCiAgICAgICAgZi53cml0ZSgnIyBIRUxQRVIgRlVOQ1RJT05TXG4nKQogICAgICAgIGYud3JpdGUoJyMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09XG5cbicpCiAgICAgICAgCiAgICAgICAgZi53cml0ZSgnZGVmIGdldF9hbGxfY29udGV4dHMoKTpcbicpCiAgICAgICAgZi53cml0ZSgnICAgICIiIlJldHVybiBhbGwgY29udGV4dHMgYXMgYSBkaWN0aW9uYXJ5LiIiIlxuJykKICAgICAgICBmLndyaXRlKCcgICAgcmV0dXJuIHtcbicpCiAgICAgICAgZm9yIHRpdGxlIGluIGNvbnRleHRzLmtleXMoKToKICAgICAgICAgICAgdmFyX25hbWUgPSBjcmVhdGVfdmFyaWFibGVfbmFtZSh0aXRsZSkKICAgICAgICAgICAgZXNjYXBlZF90aXRsZSA9IHRpdGxlLnJlcGxhY2UoJyInLCAnXFwiJykKICAgICAgICAgICAgZi53cml0ZShmJyAgICAgICAgIntlc2NhcGVkX3RpdGxlfSI6IHt2YXJfbmFtZX0sXG4nKQogICAgICAgIGYud3JpdGUoJyAgICB9XG5cbicpCiAgICAgICAgCiAgICAgICAgZi53cml0ZSgnZGVmIGdldF90ZXN0X3F1ZXN0aW9ucygpOlxuJykKICAgICAgICBmLndyaXRlKCcgICAgIiIiUmV0dXJuIHRlc3QgcXVlc3Rpb25zIHdpdGggZXhwZWN0ZWQgYW5zd2Vycy4iIiJcbicpCiAgICAgICAgZi53cml0ZSgnICAgIHJldHVybiBURVNUX1FVRVNUSU9OU1xuJykKICAgIAogICAgcHJpbnQoZiJPdXRwdXQgc2F2ZWQgdG86IHtvdXRwdXRfcGF0aH0iKQoKCmRlZiBjcmVhdGVfdmFyaWFibGVfbmFtZSh0aXRsZTogc3RyKSAtPiBzdHI6CiAgICAiIiIKICAgIENyZWF0ZSBhIHZhbGlkIFB5dGhvbiB2YXJpYWJsZSBuYW1lIGZyb20gdGl0bGUuCiAgICAKICAgIEFyZ3M6CiAgICAgICAgdGl0bGU6IE9yaWdpbmFsIHRpdGxlIHN0cmluZwogICAgICAgIAogICAgUmV0dXJuczoKICAgICAgICBWYWxpZCBQeXRob24gdmFyaWFibGUgbmFtZQogICAgIiIiCiAgICAjIFJlcGxhY2Ugbm9uLWFscGhhbnVtZXJpYyBjaGFyYWN0ZXJzIHdpdGggdW5kZXJzY29yZQogICAgbmFtZSA9IHJlLnN1YihyJ1teYS16QS1aMC05XScsICdfJywgdGl0bGUpCiAgICAjIFJlbW92ZSBjb25zZWN1dGl2ZSB1bmRlcnNjb3JlcwogICAgbmFtZSA9IHJlLnN1YihyJ18rJywgJ18nLCBuYW1lKQogICAgIyBSZW1vdmUgbGVhZGluZy90cmFpbGluZyB1bmRlcnNjb3JlcwogICAgbmFtZSA9IG5hbWUuc3RyaXAoJ18nKQogICAgIyBFbnN1cmUgc3RhcnRzIHdpdGggbGV0dGVyCiAgICBpZiBuYW1lIGFuZCBuYW1lWzBdLmlzZGlnaXQoKToKICAgICAgICBuYW1lID0gJ0NPTlRFWFRfJyArIG5hbWUKICAgIGVsc2U6CiAgICAgICAgbmFtZSA9ICdDT05URVhUXycgKyBuYW1lLnVwcGVyKCkKICAgIHJldHVybiBuYW1lCgoKZGVmIG1haW4oKToKICAgICIiIk1haW4gZnVuY3Rpb24gdG8gcHJvY2VzcyB0aGUgZGF0YXNldC4iIiIKICAgICMgQ29uZmlndXJhdGlvbiAtIGF1dG8tZGV0ZWN0IGlucHV0IGZvcm1hdAogICAgSU5QVVRfSlNPTiA9ICJ0cmFpbi12Mi4wLmpzb24iCiAgICBJTlBVVF9DU1YgPSAidHJhaW4uY3N2IgogICAgT1VUUFVUX0ZJTEUgPSAicHJvY2Vzc2VkX3NxdWFkLnB5IgogICAgCiAgICAjIEdldCBzY3JpcHQgZGlyZWN0b3J5IGZvciByZWxhdGl2ZSBwYXRocwogICAgc2NyaXB0X2RpciA9IG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmFic3BhdGgoX19maWxlX18pKQogICAgb3V0cHV0X3BhdGggPSBvcy5wYXRoLmpvaW4oc2NyaXB0X2RpciwgT1VUUFVUX0ZJTEUpCiAgICAKICAgIHByaW50KCI9IiAqIDYwKQogICAgcHJpbnQoIlNRdUFEIERhdGFzZXQgUHJvY2Vzc29yIikKICAgIHByaW50KCI9IiAqIDYwKQogICAgCiAgICAjIFRyeSB0byBmaW5kIGlucHV0IGZpbGUgKEpTT04gb3IgQ1NWKQogICAganNvbl9wYXRoID0gb3MucGF0aC5qb2luKHNjcmlwdF9kaXIsIElOUFVUX0pTT04pCiAgICBjc3ZfcGF0aCA9IG9zLnBhdGguam9pbihzY3JpcHRfZGlyLCBJTlBVVF9DU1YpCiAgICAKICAgICMgTG9hZCBkYXRhIC0gcHJlZmVyIEpTT04gKG9mZmljaWFsIFNRdUFEIGZvcm1hdCkKICAgIHByaW50KCJcblsxLzNdIExvYWRpbmcgZGF0YS4uLiIpCiAgICBpZiBvcy5wYXRoLmV4aXN0cyhqc29uX3BhdGgpOgogICAgICAgIHByaW50KGYiRm91bmQgSlNPTiBmaWxlOiB7SU5QVVRfSlNPTn0iKQogICAgICAgIGRhdGEgPSBsb2FkX2pzb25fZGF0YShqc29uX3BhdGgpCiAgICBlbGlmIG9zLnBhdGguZXhpc3RzKGNzdl9wYXRoKToKICAgICAgICBwcmludChmIkZvdW5kIENTViBmaWxlOiB7SU5QVVRfQ1NWfSIpCiAgICAgICAgZGF0YSA9IGxvYWRfY3N2X2RhdGEoY3N2X3BhdGgpCiAgICBlbHNlOgogICAgICAgIHByaW50KGYi4p2MIEVycm9yOiBObyBpbnB1dCBmaWxlIGZvdW5kISIpCiAgICAgICAgcHJpbnQoZiIgICBFeHBlY3RlZDoge0lOUFVUX0pTT059IG9yIHtJTlBVVF9DU1Z9IGluIHtzY3JpcHRfZGlyfSIpCiAgICAgICAgcHJpbnQoZiJcbiAgIFRvIGRvd25sb2FkIFNRdUFEIHYyLjA6IikKICAgICAgICBwcmludChmIiAgIHdnZXQgaHR0cHM6Ly9yYWpwdXJrYXIuZ2l0aHViLmlvL1NRdUFELWV4cGxvcmVyL2RhdGFzZXQvdHJhaW4tdjIuMC5qc29uIikKICAgICAgICByZXR1cm4KICAgIAogICAgIyBQcm9jZXNzIGRhdGEKICAgIHByaW50KCJcblsyLzNdIFByb2Nlc3NpbmcgZGF0YXNldC4uLiIpCiAgICBjb250ZXh0cywgcXVlc3Rpb25zID0gcHJvY2Vzc19kYXRhc2V0KGRhdGEpCiAgICAKICAgICMgR2VuZXJhdGUgb3V0cHV0CiAgICBwcmludCgiXG5bMy8zXSBHZW5lcmF0aW5nIG91dHB1dCBmaWxlLi4uIikKICAgIGdlbmVyYXRlX291dHB1dF9maWxlKGNvbnRleHRzLCBxdWVzdGlvbnMsIG91dHB1dF9wYXRoKQogICAgCiAgICAjIFN1bW1hcnkKICAgIHByaW50KCJcbiIgKyAiPSIgKiA2MCkKICAgIHByaW50KCJTVU1NQVJZIikKICAgIHByaW50KCI9IiAqIDYwKQogICAgcHJpbnQoZiIgIC0gVW5pcXVlIHRpdGxlczoge2xlbihjb250ZXh0cyl9IikKICAgIHByaW50KGYiICAtIFRvdGFsIFFBIHBhaXJzOiB7c3VtKGxlbihxYSkgZm9yIHFhIGluIHF1ZXN0aW9ucy52YWx1ZXMoKSl9IikKICAgIHByaW50KGYiICAtIE91dHB1dCBmaWxlOiB7b3V0cHV0X3BhdGh9IikKICAgIHByaW50KCI9IiAqIDYwKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK'
    os.makedirs('data', exist_ok=True)
    with open('data/process_squad_data.py', 'wb') as f:
        f.write(base64.b64decode(script_b64))
    
    print('⚙️ Processing SQuAD data...')
    !python data/process_squad_data.py
    
    if os.path.exists('data/processed_squad.py'):
        size_mb = os.path.getsize('data/processed_squad.py') / (1024 * 1024)
        print(f'✅ SQuAD dataset generated ({size_mb:.1f} MB)')
    else:
        print('❌ Failed to generate SQuAD dataset')


## 3. Run Baseline Comparison (Sample Essays)

This cell runs comparison on the Climate essay with 50 epochs.

In [ ]:
# Run baseline comparison on sample essays
!python projects/hybrid_titans/compare_baselines.py --essay climate --epochs 50

## 4. Run Baseline Comparison (SQuAD Dataset)

This cell runs comparison on the SQuAD dataset.

**Parameters:**
- `--squad`: Enable SQuAD mode
- `--titles N`: Number of titles to evaluate (default: 10)
- `--max-questions N`: Max questions per title (default: 5)
- `--epochs N`: Digestion epochs (default: 50)
- `--verbose`: Show detailed output

In [ ]:
# Run baseline comparison on SQuAD dataset
# Uncomment the line below to run

!python projects/hybrid_titans/compare_baselines.py --squad --titles 5 --max-questions 3 --epochs 20


## 5. Custom Comparison (Optional)

Run the comparison with your own parameters.

In [ ]:
# Custom parameters for sample essays
# !python projects/hybrid_titans/compare_baselines.py --essay ai --epochs 100 --topk 5

# Custom parameters for SQuAD
# !python projects/hybrid_titans/compare_baselines.py --squad --titles 20 --max-questions 10 --epochs 50 --verbose